# Step 1: Set up Colab and read the Excel file

In [103]:
!pip -q install dash plotly scipy openpyxl

import pandas as pd
import numpy as np
from google.colab import files

uploaded = files.upload()
file_name = list(uploaded.keys())[0]

raw = pd.read_excel(file_name)
print("Rows:", raw.shape[0], "| Columns:", raw.shape[1])
raw.head()

Saving Ads-1Jun2026-31Aug2026.xlsx to Ads-1Jun2026-31Aug2026 (7).xlsx
Rows: 329 | Columns: 43


,Reporting starts,Reporting ends,Ad name,Ad delivery,Ad set budget,Ad set budget type,Amount spent (MYR),Reach,Frequency,CPC (cost per link click) (MYR),...,Video plays at 50%,Video plays at 75%,Video plays at 95%,Video plays at 100%,Video average play time,Landing page views,Cost per landing page view (MYR).1,Date created,Results (initial),Results (initial) indicator
0,2026-06-01,2026-08-31,Sahiyya conversion Bofu â€“ 4,not_delivering,Using campaign budget,0,0.00,0,0.000000,NaN,...,NaN,NaN,NaN,NaN,00:00:00,NaN,NaN,2026-05-05,NaN,NaN
1,2026-06-01,2026-08-31,Sahiyya conversion Bofu 2 botol â€“ 3,not_delivering,Using campaign budget,0,0.00,0,0.000000,NaN,...,NaN,NaN,NaN,NaN,00:00:00,NaN,NaN,2026-05-05,NaN,NaN
2,2026-06-01,2026-08-31,Sahiyya conversion Bofu â€“ 2,not_delivering,Using campaign budget,0,0.00,0,0.000000,NaN,...,NaN,NaN,NaN,NaN,00:00:00,NaN,NaN,2026-05-05,NaN,NaN
3,2026-06-01,2026-08-31,Sahiyya conversion Bofu 3 botol â€“ 2,not_delivering,Using campaign budget,0,0.00,0,0.000000,NaN,...,NaN,NaN,NaN,NaN,00:00:00,NaN,NaN,2026-05-05,NaN,NaN
4,2026-06-01,2026-08-31,Purchased 7,not_delivering,Using campaign budget,0,46.59,699,1.211731,3.106,...,NaN,NaN,NaN,NaN,00:00:00,12.0,3.8825,2026-06-08,NaN,NaN


# Step 2: Clean the data

In [104]:
import re
import datetime as dt

# ---------- Settings you can edit ----------
PERIOD_START = '2026-07-01'   # default date filter (uses "Date created")
PERIOD_END   = '2026-08-31'
NAME_FIXES   = {}             # rename ugly ad names here, e.g. {'08-Jan': '8-1'}

# ---------- Helper 1: tidy one ad name ----------
def fix_ad_name(x):
    if isinstance(x, (dt.datetime, dt.date, pd.Timestamp)):
        s = f"{x.day:02d}-{x.strftime('%b')}"          # Excel turned the name into a date
    else:
        s = str(x).strip()
        try:
            s = s.encode('cp1252').decode('utf-8')      # fixes "â€“" -> "–"
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
        s = s.replace('\u2013', '-').replace('\u2014', '-')
    s = re.sub(r'\s+', ' ', s).strip()                  # remove double spaces
    return NAME_FIXES.get(s, s)

# ---------- Helper 2: play time -> seconds ----------
def to_seconds(x):
    if pd.isna(x):
        return 0.0
    if isinstance(x, dt.time):
        return x.hour * 3600 + x.minute * 60 + x.second
    if isinstance(x, (dt.timedelta, pd.Timedelta)):
        return x.total_seconds()
    if isinstance(x, (int, float, np.number)):
        return float(x)
    s = str(x).strip()
    try:
        if ':' in s:                                    # "00:00:12" style
            secs = 0.0
            for p in s.split(':'):
                secs = secs * 60 + float(p)
            return secs
        return float(s)
    except ValueError:
        return 0.0

In [105]:
# Columns that are COUNTS (missing = nothing happened, so fill with 0)
count_map = {
    'Amount spent (MYR)': 'Spend',
    'Reach': 'Reach',
    'Impressions': 'Impressions',
    'Link clicks': 'Link_Clicks',
    'Unique link clicks': 'Unique_Link_Clicks',
    '3-second video plays': 'Video_3s',
    'ThruPlays': 'ThruPlays',
    'Video plays at 25%': 'Video_25',
    'Video plays at 50%': 'Video_50',
    'Video plays at 75%': 'Video_75',
    'Video plays at 95%': 'Video_95',
    'Video plays at 100%': 'Video_100',
    'Landing page views': 'Landing_Page_Views',
    'Results': 'Results',
    'Purchases': 'Purchases',
    'Purchases conversion value': 'Purchase_Value',
    'Leads': 'Leads',
    'Contacts': 'Contacts',
    'Checkouts initiated': 'Checkouts',
    'Adds to cart': 'Adds_To_Cart',
}

clean = pd.DataFrame({
    'Ad_Name':      raw['Ad name'].map(fix_ad_name),
    'Ad_Delivery':  raw['Ad delivery'].astype(str).str.strip(),
    'Date_Created': pd.to_datetime(raw['Date created'], errors='coerce'),
})

for old, new in count_map.items():
    if old in raw.columns:
        clean[new] = pd.to_numeric(raw[old], errors='coerce').fillna(0)
    else:
        clean[new] = 0.0
        print("Column not found, filled with 0:", old)

# "Clicks (all)" is not in the export, so work it out: impressions x CTR (all)
ctr_all = pd.to_numeric(raw['CTR (all)'], errors='coerce').fillna(0)
clean['Clicks_All'] = (clean['Impressions'] * ctr_all / 100).round()
clean['Clicks_All'] = np.maximum(clean['Clicks_All'], clean['Link_Clicks'])

# Average play time -> seconds, plus two helper columns so we can average it fairly later
play_sec = raw['Video average play time'].map(to_seconds)
clean['_PT_Weight'] = np.where(play_sec > 0, clean['Impressions'], 0)
clean['_PT_Sum']    = play_sec * clean['_PT_Weight']

# Drop rows that never delivered (no impressions and no spend)
rows_before = len(clean)
clean = clean[(clean['Impressions'] > 0) | (clean['Spend'] > 0)]
print(f"Rows in Excel: {rows_before} | never delivered (dropped): {rows_before - len(clean)} | kept: {len(clean)}")

Rows in Excel: 329 | never delivered (dropped): 37 | kept: 292


In [106]:
# Check 2.1 - percent vs decimal
meta_ctr = pd.to_numeric(raw.loc[clean.index, 'CTR (link click-through rate)'], errors='coerce')
mine_ctr = clean['Link_Clicks'] / clean['Impressions'].where(clean['Impressions'] > 0) * 100
ok = meta_ctr.gt(0) & mine_ctr.notna()
print("2.1 Link CTR, mine / Meta's (should be about 1.0):", round((mine_ctr[ok] / meta_ctr[ok]).median(), 3))

# Check 2.2 - missing values
missing = clean.isna().sum()
missing = missing[missing > 0]
print("2.2 Columns with missing values:", "none" if missing.empty else missing.to_dict())

# Check 2.3 - ad names that Excel turned into dates
was_date = raw['Ad name'].map(lambda x: isinstance(x, (dt.datetime, dt.date, pd.Timestamp)))
print("2.3 Date-like ad names (now shown as DD-Mon):", sorted(set(raw.loc[was_date, 'Ad name'].map(fix_ad_name))))

# Check 2.4 - general
print("2.4 Unique ad names:", clean['Ad_Name'].nunique())
print("    Date created from", clean['Date_Created'].min().date(), "to", clean['Date_Created'].max().date())
print("    Result indicator values:", raw['Result indicator'].dropna().unique().tolist())

2.1 Link CTR, mine / Meta's (should be about 1.0): 1.0
2.2 Columns with missing values: none
2.3 Date-like ad names (now shown as DD-Mon): ['08-Apr', '08-Feb', '08-Jan', '08-Jul', '08-Jun', '08-Mar', '08-May']
2.4 Unique ad names: 152
    Date created from 2026-04-14 to 2026-08-19
    Result indicator values: ['actions:offsite_conversion.fb_pixel_purchase']


# Step 3: Metrics and filter functions

In [107]:
COUNT_COLS = ['Spend', 'Reach', 'Impressions', 'Link_Clicks', 'Unique_Link_Clicks', 'Clicks_All',
              'Video_3s', 'ThruPlays', 'Video_25', 'Video_50', 'Video_75', 'Video_95', 'Video_100',
              'Landing_Page_Views', 'Results', 'Purchases', 'Purchase_Value',
              'Leads', 'Contacts', 'Checkouts', 'Adds_To_Cart', '_PT_Weight', '_PT_Sum']

def ratio(a, b, mult=1.0):
    """a divided by b, times mult. Gives NaN (empty) when b is 0."""
    return a / b.where(b > 0) * mult

def add_metrics(d):
    """Take a table of ADDED-UP counts and work out every rate and cost."""
    d = d.copy()

    d['CPR']            = ratio(d['Spend'], d['Results'])
    d['CPM']            = ratio(d['Spend'], d['Impressions'], 1000)

    d['CTR_All']        = ratio(d['Clicks_All'], d['Impressions'], 100)
    d['CTR_Link']       = ratio(d['Link_Clicks'], d['Impressions'], 100)
    d['Unique_CTR']     = ratio(d['Unique_Link_Clicks'], d['Reach'], 100)

    d['CPC_Link']       = ratio(d['Spend'], d['Link_Clicks'])
    d['CPC']            = d['CPC_Link']

    d['LPV_Rate']       = ratio(
        d['Landing_Page_Views'], d['Link_Clicks'], 100
    )
    d['CPLPV']          = ratio(
        d['Spend'], d['Landing_Page_Views']
    )

    d['CPP']            = ratio(d['Spend'], d['Purchases'])
    d['ROAS']           = ratio(d['Purchase_Value'], d['Spend'])
    d['CPL']            = ratio(d['Spend'], d['Leads'])

    d['Frequency']      = ratio(d['Impressions'], d['Reach'])
    d['Results_per_1k'] = ratio(d['Results'], d['Impressions'], 1000)

    d['Hook_Rate']      = ratio(
        d['Video_3s'], d['Impressions'], 100
    ).where(d['Video_3s'] > 0)

    d['Hold_Rate']      = ratio(d['ThruPlays'], d['Video_3s'], 100)
    d['Finish_Rate']    = ratio(d['Video_100'], d['Video_3s'], 100)
    d['Avg_Play_Sec']   = ratio(d['_PT_Sum'], d['_PT_Weight'])

    return d

def filter_rows(df, start=None, end=None, ads=None):
    keep = pd.Series(True, index=df.index)
    if start:
        keep &= df['Date_Created'] >= pd.Timestamp(start)
    if end:
        keep &= df['Date_Created'] <= pd.Timestamp(end)
    if ads:
        keep &= df['Ad_Name'].isin(ads)
    return df[keep]

def aggregate_ads(rows):
    """One row per ad name. Counts are added first, THEN rates are worked out."""
    g = rows.groupby('Ad_Name', sort=False)
    out = g[COUNT_COLS].sum()
    out['Rows'] = g.size()
    out['Ad_Delivery'] = g['Ad_Delivery'].agg(lambda s: 'active' if (s == 'active').any() else s.iloc[0])
    out['Date_Created'] = g['Date_Created'].min()
    return add_metrics(out).reset_index()

def totals(rows):
    """One row with the grand totals of the rows you give it."""
    return add_metrics(rows[COUNT_COLS].sum().to_frame().T).iloc[0]

# ---- checks ----
period = filter_rows(clean, PERIOD_START, PERIOD_END)
ads = aggregate_ads(period)
t = totals(period)

print("Rows in period:", len(period), "| Unique ads:", len(ads))
print(f"Total spend:    RM {t['Spend']:,.2f}")
print(f"Total results:  {t['Results']:,.0f}")
print(f"Purchase value: RM {t['Purchase_Value']:,.2f}")
print(f"Overall CPR:    RM {t['CPR']:,.2f}")
print(f"Overall ROAS:   {t['ROAS']:.2f}")
hook_min, hook_max = ads['Hook_Rate'].min(), ads['Hook_Rate'].max()
print("Hook rate range (%):", round(hook_min, 1), "to", round(hook_max, 1))
ads.sort_values('Spend', ascending=False)[['Ad_Name','Spend','Results','CPR','Impressions','CTR_Link','Hook_Rate','Avg_Play_Sec']].head(5).round(2)

Rows in period: 177 | Unique ads: 93
Total spend:    RM 14,794.70
Total results:  46
Purchase value: RM 13,514.00
Overall CPR:    RM 321.62
Overall ROAS:   0.91
Hook rate range (%): 0.0 to 52.9


,Ad_Name,Spend,Results,CPR,Impressions,CTR_Link,Hook_Rate,Avg_Play_Sec
37,Bofu 3,3031.43,11.0,275.58,68571,1.16,19.89,3.00
7,Sahiyya conversion Bofu,1185.77,6.0,197.63,29552,1.01,0.03,2.66
6,Sahiyya conversion Bofu 2 botol - 2,1111.09,5.0,222.22,24494,1.18,0.02,2.00
35,Mofu 3,932.73,0.0,NaN,21269,1.27,20.35,4.64
34,Mofu 2,909.87,3.0,303.29,15429,1.78,41.05,9.37


# Step 4: Dashboard shell (sidebar + empty sections)

In [108]:
import random
from dash import Dash, dcc, html, Input, Output, ctx

SECTIONS = [
    ('overview',   '1  Overview'),
    ('table',      '2  Data Table'),
    ('video',      '3  Video Creative'),
    ('lead',       '4  Lead Analysis'),
    ('conversion', '5  Conversion'),
]
TITLES = {
    'overview':   'Overview',
    'table':      'Data Table',
    'video':      'Video Creative Analysis',
    'lead':       'Lead Analysis',
    'conversion': 'Conversion Analysis',
}
# Which filters each section shows
FILTERS_ON = {
    'overview':   ('date', 'ad'),
    'table':      ('date',),
    'video':      ('ad',),
    'lead':       ('date', 'ad'),
    'conversion': ('date', 'ad'),
}
# Empty boxes: one per item you asked for (we fill these in later steps)
PLACEHOLDERS = {
    'overview': ['KPI cards: amount spent, results, CPR, reach, impressions',
                 'Insight sentences',
                 'Total spend by campaign (ad name)', 'Video funnel',
                 'CPR vs results per ad', 'Results by campaign'],
    'table':    ['Data table: ad name, spent, results, CPR, impressions, CPM, link clicks, '
                 'clicks (all), unique link clicks, CTR link click %, CPC link click, ad tier'],
    'video':    ['KPI cards: hook rate %, hold rate %', 'Insight sentences',
                 'Hook rate % by ad', 'Hold rate % by ad', 'Finish rate % by ad',
                 'Video funnel', 'Avg play time by ad', 'Hook rate vs result',
                 'Creative fatigue flags', 'Correlation matrix'],
    'lead':     ['KPI cards: no. of leads, CPL', 'Insight sentences',
                 'Flow funnel by ad (hook, hold, CTR, result)',
                 'Hook rate vs results per 1k impressions',
                 'Data performance table (parameters vs ad name)',
                 'Summary table with winning ad score + Scale / Watch / Kill'],
    'conversion': ['KPI cards: spend, CPP, purchases, CTR, LPV rate, purchase value, ROAS',
                   'Insight sentences (CPP, CTR, LPV)',
                   'Summary table: ad name, CPM, purchases, CPP, CVR score',
                   'Bar chart: switch between CPP / unique CTR / LPV rate',
                   'Compare two ads (is the gap real?)', 'Spend forecast', 'Budget what-if simulator'],
}

SIDEBAR_W = '230px'
NAVY, LIGHT = '#1f2937', '#f3f4f6'

def nav_style(active):
    return {'display': 'block', 'width': '100%', 'textAlign': 'left', 'padding': '12px 14px',
            'marginBottom': '6px', 'border': 'none', 'borderRadius': '8px', 'cursor': 'pointer',
            'fontSize': '15px', 'color': 'white',
            'backgroundColor': '#3b82f6' if active else 'transparent'}

all_ads = sorted(clean['Ad_Name'].unique())
date_min = min(clean['Date_Created'].min(), pd.Timestamp(PERIOD_START)).date()
date_max = max(clean['Date_Created'].max(), pd.Timestamp(PERIOD_END)).date()

app = Dash(__name__, suppress_callback_exceptions=True)

app.layout = html.Div(style={'fontFamily': 'Arial, sans-serif', 'backgroundColor': LIGHT, 'minHeight': '100vh'}, children=[
    dcc.Store(id='section', data='overview'),

    # ----- Side menu -----
    html.Div(style={'position': 'fixed', 'top': 0, 'left': 0, 'bottom': 0, 'width': SIDEBAR_W,
                    'backgroundColor': NAVY, 'padding': '24px 14px', 'boxSizing': 'border-box'}, children=[
        html.H3('AFYAA Meta Ads Performance', style={'color': 'white', 'margin': '0 0 24px 8px'}),
        *[html.Button(label, id=f'nav-{key}', n_clicks=0, style=nav_style(key == 'overview'))
          for key, label in SECTIONS],
    ]),

    # ----- Main area -----
    html.Div(style={'marginLeft': SIDEBAR_W, 'padding': '24px'}, children=[
        html.H2(id='page-title', style={'margin': '0 0 16px 0', 'color': NAVY}),

        # Filter bar
        html.Div(style={'display': 'flex', 'gap': '20px', 'alignItems': 'flex-end', 'flexWrap': 'wrap',
                        'backgroundColor': 'white', 'padding': '14px 18px', 'borderRadius': '10px',
                        'marginBottom': '20px'}, children=[
            html.Div(id='date-box', children=[
                html.Div('Date created', style={'fontWeight': 'bold', 'marginBottom': '4px'}),
                dcc.DatePickerRange(id='date-range', min_date_allowed=date_min, max_date_allowed=date_max,
                                    start_date=PERIOD_START, end_date=PERIOD_END,
                                    display_format='DD MMM YYYY'),
            ]),
            html.Div(id='ad-box', style={'minWidth': '360px'}, children=[
                html.Div('Ad name', style={'fontWeight': 'bold', 'marginBottom': '4px'}),
                dcc.Dropdown(id='ad-filter', options=all_ads, value=[], multi=True,
                             placeholder='All ads'),
            ]),
        ]),

        html.Div(id='page'),
    ]),
])

# ----- Remember which menu button was clicked -----
@app.callback(Output('section', 'data'),
              [Input(f'nav-{k}', 'n_clicks') for k, _ in SECTIONS],
              prevent_initial_call=True)
def pick_section(*clicks):
    return ctx.triggered_id.replace('nav-', '')

# ----- Work out which rows this section should use -----
def rows_for(section, start, end, ads):
    use_date = 'date' in FILTERS_ON[section]
    use_ad   = 'ad' in FILTERS_ON[section]
    return filter_rows(clean,
                       start if use_date else PERIOD_START,
                       end   if use_date else PERIOD_END,
                       ads   if use_ad   else None)

# ----- Draw the page (this is the ONE place every chart will get its data) -----
def build_section(section, rows):
    t = totals(rows)
    n_ads = rows['Ad_Name'].nunique()
    status = html.Div(style={'backgroundColor': 'white', 'padding': '14px 18px', 'borderRadius': '10px',
                             'marginBottom': '16px', 'borderLeft': '5px solid #3b82f6'}, children=[
        html.Div('Data this section is using right now', style={'fontWeight': 'bold', 'marginBottom': '6px'}),
        html.Div(f"Rows: {len(rows):,}   |   Ads: {n_ads:,}   |   Spend: RM {t['Spend']:,.2f}   |   "
                 f"Results: {t['Results']:,.0f}"),
    ])
    boxes = [html.Div(text, style={'border': '2px dashed #9ca3af', 'borderRadius': '10px', 'padding': '28px 16px',
                                   'color': '#6b7280', 'backgroundColor': 'white', 'minHeight': '70px'})
             for text in PLACEHOLDERS[section]]
    grid = html.Div(boxes, style={'display': 'grid', 'gridTemplateColumns': 'repeat(auto-fit, minmax(320px, 1fr))',
                                  'gap': '16px'})
    return html.Div([status, grid])

@app.callback(
    Output('page', 'children'),
    Output('page-title', 'children'),
    Output('date-box', 'style'),
    Output('ad-box', 'style'),
    *[Output(f'nav-{k}', 'style') for k, _ in SECTIONS],
    Input('section', 'data'),
    Input('date-range', 'start_date'),
    Input('date-range', 'end_date'),
    Input('ad-filter', 'value'),
)
def render(section, start, end, ads):
    rows = rows_for(section, start, end, ads)
    show = lambda name, extra={}: {**extra, 'display': 'block' if name in FILTERS_ON[section] else 'none'}
    return (build_section(section, rows),
            TITLES[section],
            show('date'),
            show('ad', {'minWidth': '360px'}),
            *[nav_style(k == section) for k, _ in SECTIONS])


In [109]:
PORT = random.randint(8100, 8900)     # new port every run, so old apps never clash
app.run(debug=False, port=PORT, jupyter_height=900)

<IPython.core.display.Javascript object>

# Step 5: Overview section

In [110]:
# ============ STEP 5: OVERVIEW SECTION ============
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------- 5A. Settings ----------
TOP_N     = 15     # how many ads to show in the charts
MIN_SPEND = 50     # RM. An ad with less spend than this is "too early to judge"

BLUE, GREEN, AMBER, RED, GREY = '#3b82f6', '#16a34a', '#f59e0b', '#dc2626', '#6b7280'

# ---------- 5B. Small helpers ----------
def rm(x):
    return f"RM {x:,.2f}"

def vs_bench(x, bench):
    """'31% above' / '12% below' the benchmark."""
    diff = (x - bench) / bench * 100
    return f"{abs(diff):.0f}% {'above' if diff >= 0 else 'below'}"

def empty_fig(msg, height=380):
    fig = go.Figure()
    fig.add_annotation(text=msg, x=0.5, y=0.5, xref='paper', yref='paper', showarrow=False,
                       font=dict(size=15, color=GREY))
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    fig.update_layout(height=height, template='plotly_white', margin=dict(l=10, r=10, t=40, b=10))
    return fig

def card(title, child, note=None):
    kids = [html.Div(title, style={'fontWeight': 'bold', 'marginBottom': '8px', 'color': NAVY}), child]
    if note:
        kids.append(html.Div(note, style={'fontSize': '12px', 'color': GREY, 'marginTop': '6px'}))
    return html.Div(kids, style={'backgroundColor': 'white', 'padding': '16px', 'borderRadius': '10px'})

def graph(fig):
    return dcc.Graph(figure=fig, config={'displayModeBar': False})

def kpi_card(label, value, note=None):
    kids = [html.Div(label, style={'fontSize': '13px', 'color': GREY}),
            html.Div(value, style={'fontSize': '26px', 'fontWeight': 'bold', 'color': NAVY, 'marginTop': '4px'})]
    if note:
        kids.append(html.Div(note, style={'fontSize': '11px', 'color': GREY, 'marginTop': '4px'}))
    return html.Div(kids, style={'backgroundColor': 'white', 'padding': '16px', 'borderRadius': '10px',
                                 'borderTop': f'4px solid {BLUE}'})

def insight_box(items):
    """items = list of (level, sentence). level: good / warn / info"""
    colour = {'good': GREEN, 'warn': AMBER, 'info': BLUE}
    lis = [html.Li([html.Span('● ', style={'color': colour[lvl]}), text], style={'marginBottom': '8px'})
           for lvl, text in items]
    return html.Div([html.Div('Insights', style={'fontWeight': 'bold', 'marginBottom': '8px', 'color': NAVY}),
                     html.Ul(lis, style={'listStyle': 'none', 'padding': 0, 'margin': 0})],
                    style={'backgroundColor': 'white', 'padding': '16px', 'borderRadius': '10px',
                           'marginBottom': '16px', 'borderLeft': f'5px solid {AMBER}'})

# ---------- 5C. Insight sentences ----------
def overview_insights(view, peers_tbl, tot):
    """view = ad table of the selected ads. peers_tbl = ad table of ALL ads in the date range."""
    if view.empty or tot['Spend'] <= 0:
        return [('info', 'No data for these filters.')]

    out = []
    peer_tot = totals_from_table(peers_tbl)
    bench = peer_tot['CPR']                      # overall CPR of all ads in this date range
    bench_ok = pd.notna(bench)

    # a) biggest spender
    top = view.sort_values('Spend', ascending=False).iloc[0]
    share = top['Spend'] / tot['Spend'] * 100
    s = f"The biggest spender is '{top['Ad_Name']}': {rm(top['Spend'])} ({share:.0f}% of spend)"
    if top['Results'] > 0:
        s += f", which gave {top['Results']:,.0f} result(s) at a CPR of {rm(top['CPR'])}"
        if bench_ok:
            s += f" ({vs_bench(top['CPR'], bench)} the overall CPR of {rm(bench)})"
    else:
        s += ", and it has no results yet"
    out.append(('warn' if top['Results'] == 0 else 'info', s + '.'))

    # b) best and c) worst CPR (only ads with enough spend and at least 1 result)
    judged = view[(view['Spend'] >= MIN_SPEND) & (view['Results'] > 0)]
    if len(judged) >= 1 and bench_ok:
        best = judged.sort_values('CPR').iloc[0]
        s = (f"'{best['Ad_Name']}' has the lowest CPR: {rm(best['CPR'])}, "
             f"{vs_bench(best['CPR'], bench)} the overall CPR")
        if best['Results'] < 3:
            s += f" (only {best['Results']:.0f} result(s), so treat it with care)"
        out.append(('good', s + '.'))
    if len(judged) >= 2 and bench_ok:
        worst = judged.sort_values('CPR').iloc[-1]
        if worst['CPR'] > bench:
            out.append(('warn', f"'{worst['Ad_Name']}' has the highest CPR: {rm(worst['CPR'])}, "
                                f"{vs_bench(worst['CPR'], bench)} the overall CPR - consider pausing or fixing it."))

    # d) money spent with nothing to show
    waste = view[(view['Spend'] >= MIN_SPEND) & (view['Results'] == 0) & (view['Leads'] == 0)]
    if len(waste) > 0 and len(view) > 1:
        w = waste.sort_values('Spend', ascending=False).iloc[0]
        out.append(('warn', f"{len(waste)} ad(s) spent {rm(waste['Spend'].sum())} "
                            f"({waste['Spend'].sum() / tot['Spend'] * 100:.0f}% of spend) with no results or leads. "
                            f"Biggest: '{w['Ad_Name']}' ({rm(w['Spend'])})."))

    # e) video
    vid = view[view['Video_3s'] > 0]
    if len(vid) > 0:
        v3, imp, v100 = vid['Video_3s'].sum(), vid['Impressions'].sum(), vid['Video_100'].sum()
        out.append(('info', f"On video ads, {v3 / imp * 100:.1f}% of impressions turn into a 3-second view, "
                            f"and {v100 / v3 * 100:.1f}% of those viewers watch to the end."))

    # f) concentration
    if len(view) >= 5:
        top3 = view.sort_values('Spend', ascending=False).head(3)['Spend'].sum() / tot['Spend'] * 100
        if top3 >= 50:
            out.append(('info', f"The top 3 ads take {top3:.0f}% of the spend."))

    # g) low-data warning
    small = (view['Spend'] < MIN_SPEND).sum()
    if small > 0:
        out.append(('info', f"{small} of {len(view)} ads spent less than RM {MIN_SPEND}, "
                            f"so their numbers are too small to judge."))
    return out

def totals_from_table(tbl):
    """Grand totals from an ad table (add the counts, then divide)."""
    return add_metrics(tbl[COUNT_COLS].sum().to_frame().T).iloc[0]

# ---------- 5D. Charts ----------
def fig_spend_by_ad(view):
    d = view[view['Spend'] > 0].sort_values('Spend', ascending=False)
    if d.empty:
        return empty_fig('No spend for these filters')
    top = d.head(TOP_N)
    fig = go.Figure(go.Bar(x=top['Spend'], y=top['Ad_Name'], orientation='h', marker_color=BLUE,
                           text=[f"RM {v:,.0f}" for v in top['Spend']], textposition='outside',
                           hovertemplate='%{y}<br>RM %{x:,.2f}<extra></extra>'))
    fig.update_yaxes(autorange='reversed', automargin=True)
    fig.update_xaxes(title='Amount spent (RM)')
    fig.update_layout(template='plotly_white', height=420, margin=dict(l=10, r=40, t=30, b=40),
                      title=dict(text=f"Top {len(top)} of {len(d)} ads", font=dict(size=13, color=GREY)))
    return fig

def fig_video_funnel(view):
    vid = view[view['Video_3s'] > 0]
    if vid.empty:
        return empty_fig('No video plays for these filters')
    imp, v3 = vid['Impressions'].sum(), vid['Video_3s'].sum()
    stages = [('Impressions (video ads)', imp), ('3-second plays', v3),
              ('Played to 25%', vid['Video_25'].sum()), ('Played to 50%', vid['Video_50'].sum()),
              ('Played to 75%', vid['Video_75'].sum()), ('Played to 95%', vid['Video_95'].sum()),
              ('Played to 100%', vid['Video_100'].sum())]
    text = [f"{imp:,.0f}", f"{v3:,.0f}  ({v3 / imp * 100:.1f}% of impressions)"]
    text += [f"{v:,.0f}  ({v / v3 * 100:.0f}% of 3-sec plays)" for _, v in stages[2:]]
    fig = go.Figure(go.Funnel(y=[s for s, _ in stages], x=[v for _, v in stages], text=text,
                              textinfo='text', marker=dict(color=BLUE)))
    fig.update_layout(template='plotly_white', height=420, margin=dict(l=10, r=10, t=30, b=10))
    return fig

def fig_cpr_vs_results(view):
    d = view[view['Results'] > 0].sort_values('Results', ascending=False).head(TOP_N)
    if d.empty:
        return empty_fig('No results for these filters')
    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Bar(x=d['Ad_Name'], y=d['Results'], name='Results', marker_color=BLUE), secondary_y=False)
    fig.add_trace(go.Scatter(x=d['Ad_Name'], y=d['CPR'], name='CPR (RM)', mode='lines+markers',
                             line=dict(color=RED)), secondary_y=True)
    fig.update_yaxes(title_text='Results', secondary_y=False)
    fig.update_yaxes(title_text='CPR (RM)', secondary_y=True, showgrid=False)
    fig.update_xaxes(tickangle=-35, automargin=True)
    fig.update_layout(template='plotly_white', height=420, margin=dict(l=10, r=10, t=30, b=10),
                      legend=dict(orientation='h', y=1.12))
    return fig

def fig_results_by_ad(view):
    d = view[view['Results'] > 0].sort_values('Results', ascending=False)
    if d.empty:
        return empty_fig('No results for these filters')
    top = d.head(TOP_N)
    share = top['Results'] / d['Results'].sum() * 100
    fig = go.Figure(go.Bar(x=top['Results'], y=top['Ad_Name'], orientation='h', marker_color=GREEN,
                           text=[f"{r:,.0f}  ({s:.0f}%)" for r, s in zip(top['Results'], share)],
                           textposition='outside', hovertemplate='%{y}<br>%{x:,.0f} results<extra></extra>'))
    fig.update_yaxes(autorange='reversed', automargin=True)
    fig.update_xaxes(title='Results')
    fig.update_layout(template='plotly_white', height=420, margin=dict(l=10, r=60, t=30, b=40),
                      title=dict(text=f"Top {len(top)} of {len(d)} ads with results (% = share of results)",
                                 font=dict(size=13, color=GREY)))
    return fig

# ---------- 5E. Put the Overview page together ----------
def build_overview(rows, peers):

    view = aggregate_ads(rows)
    peers_tbl = aggregate_ads(peers)

    # Calculate totals from the selected rows
    tot = totals(rows)

    kpis = html.Div([
        # Existing KPIs
        kpi_card(
            'Amount spent',
            rm(tot['Spend'])
        ),
        kpi_card(
            'Total results',
            f"{tot['Results']:,.0f}"
        ),
        kpi_card(
            'Overall CPR',
            rm(tot['CPR']) if pd.notna(tot['CPR']) else '-'
        ),
        kpi_card(
            'Total reach',
            f"{tot['Reach']:,.0f}",
            'Added across ads, so one person may be counted twice'
        ),
        kpi_card(
            'Impressions',
            f"{tot['Impressions']:,.0f}"
        ),

        # New marketing metrics
        kpi_card(
            'CPM',
            rm(tot['CPM']) if pd.notna(tot['CPM']) else '-',
            'Cost per 1,000 impressions'
        ),
        kpi_card(
            'CPC',
            rm(tot['CPC']) if pd.notna(tot['CPC']) else '-',
            'Cost per link click'
        ),
        kpi_card(
            'CTR Link',
            f2(tot['CTR_Link'], '%'),
            'Link clicks / impressions'
        ),
        kpi_card(
            'CPLPV',
            rm(tot['CPLPV']) if pd.notna(tot['CPLPV']) else '-',
            'Cost per landing page view'
        ),
        kpi_card(
            'CPP',
            rm(tot['CPP']) if pd.notna(tot['CPP']) else '-',
            'Cost per purchase'
        ),
        kpi_card(
            'ROAS',
            f2(tot['ROAS']),
            'Purchase value / spend'
        ),

    ], style={
        'display': 'grid',
        'gridTemplateColumns': 'repeat(auto-fit, minmax(190px, 1fr))',
        'gap': '14px',
        'marginBottom': '16px'
    })

    two = {
        'display': 'grid',
        'gridTemplateColumns': 'repeat(auto-fit, minmax(420px, 1fr))',
        'gap': '16px',
        'marginBottom': '16px'
    }

    return html.Div([
        kpis,

        insight_box(
            overview_insights(view, peers_tbl, tot)
        ),

        html.Div([
            card(
                'Total spend by campaign (ad name)',
                graph(fig_spend_by_ad(view))
            ),
            card(
                'Video funnel',
                graph(fig_video_funnel(view))
            )
        ], style=two),

        html.Div([
            card(
                'CPR vs results per ad',
                graph(fig_cpr_vs_results(view))
            ),
            card(
                'Results by campaign',
                graph(fig_results_by_ad(view))
            )
        ], style=two),
    ])

# ---------- 5F. Plug it into the dashboard ----------
def rows_for(section, start, end, ads):
    """Now returns TWO tables: the rows to show, and all ads in the same date range (for comparing)."""
    use_date = 'date' in FILTERS_ON[section]
    use_ad   = 'ad'   in FILTERS_ON[section]
    peers = filter_rows(clean, start if use_date else PERIOD_START, end if use_date else PERIOD_END)
    rows  = filter_rows(peers, ads=ads if use_ad else None)
    return rows, peers

if 'placeholder_section' not in globals():
    placeholder_section = build_section          # keep the grey boxes for sections not built yet

SECTION_BUILDERS = {'overview': build_overview}

def build_section(section, rows_and_peers):
    rows, peers = rows_and_peers
    if section in SECTION_BUILDERS:
        return SECTION_BUILDERS[section](rows, peers)
    return placeholder_section(section, rows)

# Step 6: Winning ad score, Data Table, text filter box

In [111]:
# ============ STEP 6: WINNING SCORE, DATA TABLE, TEXT FILTER ============
from dash import dash_table, State, no_update
from dash.dash_table.Format import Format, Group, Scheme

# ---------- 6A. Score settings (change these to suit your business) ----------
WEIGHTS            = {'ROAS': 35, 'CPP': 30, 'CTR_Link': 20, 'Hook_Rate': 15}   # add up to 100
SCALE_MIN          = 65      # score at or above this -> Scale
KILL_MAX           = 35      # score at or below this -> Kill
KILL_NO_SALE_SPEND = 150     # never Kill a no-sale ad before it has spent at least this much (RM)
NO_SALE_MULT       = 2       # ...and not before it has spent this many times the average cost per purchase
BREAKEVEN_ROAS     = 1.0     # an ad with purchases needs at least this ROAS to be Scale
MIN_EVIDENCE       = 2       # an ad needs at least this many purchases + leads to be Scale
# (MIN_SPEND from Step 5 is also used: ads that spent less are always Watch)

# ---------- 6B. The winning ad score ----------
def pct_score(s, higher_is_better=True):
    """Rank the values from 0 (worst) to 100 (best). Empty values stay empty."""
    s = s.astype(float)
    n = s.notna().sum()
    if n == 0:
        return s
    if n == 1:
        return s.where(s.isna(), 50.0)
    rank = s.rank(method='average', ascending=higher_is_better)   # smallest = rank 1 when ascending
    return (rank - 1) / (n - 1) * 100

def score_ads(tbl):
    """Add Win_Score, Ad_Tier and Tier_Reason to an ad table (one row per ad)."""
    t = tbl.copy()
    if t.empty:
        for c in ['Win_Score', 'Ad_Tier', 'Tier_Reason']:
            t[c] = pd.Series(dtype='float' if c == 'Win_Score' else 'object')
        return t

    enough  = t['Spend'] >= MIN_SPEND                            # enough money spent to judge
    lead_ad = (t['Leads'] > 0) & (t['Purchases'] == 0)           # lead ads are judged on CPL, not CPP / ROAS
    no_purch = (t['Purchases'] == 0) & ~lead_ad
    no_sale = (t['Purchases'] == 0) & (t['Leads'] == 0) & (t['Results'] == 0)

    # ---- 0 to 100 score for each part ----
    parts = {}
    parts['ROAS'] = pct_score(t['ROAS'].where(enough & ~lead_ad), True)
    cpp_score  = pct_score(t['CPP'].where(enough & ~lead_ad), False)
    cpl_score  = pct_score(t['CPL'].where(enough & lead_ad), False)      # lead ads use CPL in the CPP slot
    parts['CPP'] = cpp_score.fillna(cpl_score).mask(enough & no_purch, 0.0)   # spent, no purchase = worst
    parts['CTR_Link']  = pct_score(t['CTR_Link'].where(enough), True)
    parts['Hook_Rate'] = pct_score(t['Hook_Rate'].where(enough), True)
    S = pd.DataFrame(parts)
    w = pd.Series(WEIGHTS)[S.columns]
    have = S.notna()
    t['Win_Score'] = (S.fillna(0) * w).sum(axis=1) / (have * w).sum(axis=1).replace(0, np.nan)

    # ---- how much must a no-sale ad spend before we call it a Kill? ----
    avg_cpp = t['Spend'].sum() / t['Purchases'].sum() if t['Purchases'].sum() > 0 else np.nan
    limit = KILL_NO_SALE_SPEND if pd.isna(avg_cpp) else max(KILL_NO_SALE_SPEND, NO_SALE_MULT * avg_cpp)

    def decide(r):
        if r['Spend'] < MIN_SPEND or pd.isna(r['Win_Score']):
            return 'Watch', f"Too early: spent under RM {MIN_SPEND}"
        if r['Purchases'] == 0 and r['Leads'] == 0 and r['Results'] == 0:
            if r['Spend'] >= limit:
                return 'Kill', f"Spent RM {r['Spend']:,.0f} (limit RM {limit:,.0f}) with no purchase or lead"
            return 'Watch', f"No purchase or lead yet, but spend is under RM {limit:,.0f}, so too soon to judge"
        events = r['Purchases'] + r['Leads']
        sc = r['Win_Score']
        if sc >= SCALE_MIN:
            if events < MIN_EVIDENCE:
                return 'Watch', f"Score {sc:.0f} but only {events:.0f} purchase/lead so far"
            if r['Purchases'] > 0 and r['ROAS'] < BREAKEVEN_ROAS:
                return 'Watch', f"Score {sc:.0f} but ROAS {r['ROAS']:.2f} is under {BREAKEVEN_ROAS}"
            extra = f" and ROAS {r['ROAS']:.2f}" if r['Purchases'] > 0 else f" and CPL {rm(r['CPL'])}"
            return 'Scale', f"Score {sc:.0f}{extra}"
        if sc <= KILL_MAX:
            return 'Kill', f"Score {sc:.0f} (bottom group)"
        return 'Watch', f"Score {sc:.0f} (between the Kill and Scale lines)"

    decided = t.apply(decide, axis=1, result_type='expand')
    t['Ad_Tier'], t['Tier_Reason'] = decided[0], decided[1]
    return t

def scored_view(rows, peers):
    """Ad table for the selected rows, with scores worked out against ALL ads in the date range."""
    peers_scored = score_ads(aggregate_ads(peers))
    keep = peers_scored[['Ad_Name', 'Win_Score', 'Ad_Tier', 'Tier_Reason']]
    view = aggregate_ads(rows).merge(keep, on='Ad_Name', how='left')
    return view, peers_scored

# ---------- 6C. Data Table section ----------
def money(label, key):  return {'name': label, 'id': key, 'type': 'numeric', 'format': Format(precision=2, scheme=Scheme.fixed, group=Group.yes)}
def whole(label, key):  return {'name': label, 'id': key, 'type': 'numeric', 'format': Format(precision=0, scheme=Scheme.fixed, group=Group.yes)}
def text(label, key):   return {'name': label, 'id': key}

TABLE_COLUMNS = [text('Ad name', 'Ad_Name'), money('Amount spent (RM)', 'Spend'), whole('Actual results', 'Results'),
                 money('CPR (RM)', 'CPR'), whole('Impressions', 'Impressions'), money('CPM (RM)', 'CPM'),
                 whole('Link clicks', 'Link_Clicks'), whole('Clicks (all)', 'Clicks_All'),
                 whole('Unique link clicks', 'Unique_Link_Clicks'), money('CTR link click (%)', 'CTR_Link'),
                 money('CPC link click (RM)', 'CPC_Link'), whole('Score', 'Win_Score'),
                 text('Ad tier', 'Ad_Tier'), text('Why this tier', 'Tier_Reason')]

TIER_STYLES = [{'if': {'filter_query': f'{{Ad_Tier}} = "{tier}"', 'column_id': 'Ad_Tier'},
                'backgroundColor': bg, 'color': fg, 'fontWeight': 'bold'}
               for tier, bg, fg in [('Scale', '#dcfce7', '#166534'), ('Watch', '#fef3c7', '#92400e'),
                                    ('Kill', '#fee2e2', '#991b1b')]]

def tier_insights(view):
    if view.empty:
        return [('info', 'No data for these filters.')]
    out = []
    n = len(view)
    c = view['Ad_Tier'].value_counts()
    early = (view['Spend'] < MIN_SPEND).sum()
    line = f"Of {n} ads: {c.get('Scale', 0)} Scale, {c.get('Watch', 0)} Watch, {c.get('Kill', 0)} Kill."
    if early > 0:
        line += f" {early} of the Watch ads have spent under RM {MIN_SPEND}, so it is too early to judge them."
    out.append(('info', line))
    sc = view[view['Ad_Tier'] == 'Scale'].sort_values('Win_Score', ascending=False).head(3)
    if len(sc):
        names = '; '.join(f"'{r.Ad_Name}' ({r.Tier_Reason})" for r in sc.itertuples())
        out.append(('good', f"Best ads to scale: {names}."))
    else:
        out.append(('info', "No ad passes the Scale rules right now."))
    k = view[view['Ad_Tier'] == 'Kill']
    if len(k):
        big = k.sort_values('Spend', ascending=False).iloc[0]
        out.append(('warn', f"The {len(k)} Kill ad(s) spent {rm(k['Spend'].sum())} "
                            f"({k['Spend'].sum() / view['Spend'].sum() * 100:.0f}% of spend). "
                            f"Biggest: '{big['Ad_Name']}' ({rm(big['Spend'])}) - {big['Tier_Reason'].lower()}."))
    return out

def build_data_table(rows, peers):
    view, _ = scored_view(rows, peers)
    view = view.sort_values('Spend', ascending=False)
    shown = view[[c['id'] for c in TABLE_COLUMNS]].round(2)
    data = shown.astype(object).where(shown.notna(), None).to_dict('records')

    table = dash_table.DataTable(
        id='data-table', columns=TABLE_COLUMNS, data=data, page_size=15, sort_action='native',
        export_format='csv', style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'padding': '8px', 'fontFamily': 'Arial', 'fontSize': '13px',
                    'minWidth': '90px', 'maxWidth': '280px', 'whiteSpace': 'normal'},
        style_cell_conditional=[{'if': {'column_id': 'Ad_Name'}, 'minWidth': '200px'},
                                {'if': {'column_id': 'Tier_Reason'}, 'minWidth': '230px'}],
        style_header={'fontWeight': 'bold', 'backgroundColor': '#f3f4f6'},
        style_data_conditional=TIER_STYLES)

    how = html.Details([
        html.Summary('How the score and tier are worked out', style={'cursor': 'pointer', 'fontWeight': 'bold'}),
        html.Ul([
            html.Li("Score (0 to 100) = ranking of each ad against the other ads in this date range that spent at least "
                    f"RM {MIN_SPEND}. Weights: " + ', '.join(f"{k.replace('_', ' ')} {v}%" for k, v in WEIGHTS.items())
                    + ". For CPP, lower is better. Metrics an ad does not have (for example hook rate on an image ad) "
                      "are skipped and the other weights are scaled up."),
            html.Li(f"Scale: score of {SCALE_MIN} or more, at least {MIN_EVIDENCE} purchases/leads, and ROAS of at least "
                    f"{BREAKEVEN_ROAS} if the ad has purchases."),
            html.Li(f"Kill: score of {KILL_MAX} or less. Or spent {NO_SALE_MULT}x the average cost per purchase "
                    f"(never less than RM {KILL_NO_SALE_SPEND}) with no purchase and no lead. An ad that has not had "
                    "that much chance yet stays Watch."),
            html.Li(f"Watch: everything else, including any ad that spent under RM {MIN_SPEND}."),
            html.Li("Lead ads (leads but no purchases) use CPL instead of CPP and are not judged on ROAS."),
            html.Li("Tags are a guide. Most ads have very few purchases, so small gaps can be luck."),
        ], style={'fontSize': '13px', 'color': GREY})], style={'marginTop': '14px'})

    return html.Div([insight_box(tier_insights(view)),
                     card('Ad performance table', html.Div([table, how]),
                          note='Click a column name to sort. Use the Export button to download as CSV.')])

# Data Table now also shows the Ad name filter, so the text filter box (6D) can work there too
FILTERS_ON['table'] = ('date', 'ad')
SECTION_BUILDERS['table'] = build_data_table

# Register the callback only once per app (so re-running this cell does not crash)
if not any('text-filter-msg.children' in key for key in app.callback_map):
    @app.callback(Output('ad-filter', 'value'), Output('text-filter-msg', 'children'),
                  Input('text-filter', 'value'),
                  State('section', 'data'), State('date-range', 'start_date'), State('date-range', 'end_date'),
                  prevent_initial_call=True)
    def use_text_filter(text_in, section, start, end):
        return text_filter_logic(text_in, section, start, end)

In [112]:
# ============ STEP 6E: METRIC COMPARE FILTER (dropdown + number) ============

# label shown -> column in the ad table
METRIC_ROWS = [('CPM', 'CPM'), ('CPC', 'CPC_Link'), ('CTR', 'CTR_Link'),
               ('CPLPV', 'CPLPV'), ('CPP', 'CPP'), ('ROAS', 'ROAS')]
OP_OPTIONS = [{'label': '>', 'value': '>'},
              {'label': '<',    'value': '<'},
              {'label': '=',        'value': '=='}]

def _metric_row(label):
    return html.Div(style={'display': 'flex', 'gap': '8px', 'alignItems': 'center', 'marginBottom': '6px'}, children=[
        html.Div(label, style={'width': '60px', 'fontWeight': 'bold'}),
        dcc.Dropdown(id=f'mf-op-{label}', options=OP_OPTIONS, placeholder='Skip', style={'width': '170px'}),
        dcc.Input(id=f'mf-val-{label}', type='number', placeholder='value',
                  style={'width': '90px', 'padding': '8px', 'boxSizing': 'border-box'}),
    ])

# Add the panel to the filter bar (only once)
_main = app.layout.children[2]
_bar = _main.children[1]
if not any(getattr(c, 'id', None) == 'metric-box' for c in _bar.children):
    _bar.children.append(html.Div(id='metric-box', style={'flexBasis': '100%'}, children=[
        html.Div('Compare metrics (leave a metric on "Skip" to ignore it)',
                 style={'fontWeight': 'bold', 'marginBottom': '6px'}),
        html.Div([_metric_row(lbl) for lbl, _ in METRIC_ROWS],
                 style={'display': 'grid', 'gridTemplateColumns': 'repeat(auto-fit, minmax(340px, 1fr))',
                        'columnGap': '20px'}),
        html.Div(style={'display': 'flex', 'gap': '10px', 'alignItems': 'center', 'marginTop': '6px'}, children=[
            html.Button('Apply', id='mf-apply', n_clicks=0,
                        style={'padding': '8px 18px', 'border': 'none', 'borderRadius': '6px',
                               'backgroundColor': '#3b82f6', 'color': 'white', 'cursor': 'pointer'}),
            html.Button('Clear', id='mf-clear', n_clicks=0,
                        style={'padding': '8px 18px', 'border': '1px solid #d1d5db', 'borderRadius': '6px',
                               'backgroundColor': 'white', 'cursor': 'pointer'}),
            html.Div(id='metric-filter-msg', style={'fontSize': '12px', 'color': GREY}),
        ]),
    ]))

def metric_filter_logic(ops, vals, section, start, end):
    conds = []
    for (label, col), op, val in zip(METRIC_ROWS, ops, vals):
        if not op:
            continue                                   # Skip = ignore this metric
        if val is None:
            return no_update, f"{label}: please type a number."
        conds.append((label, col, op, float(val)))
    if not conds:
        return no_update, "Pick at least one metric first."

    _, peers = rows_for(section, start, end, None)     # all ads in the date range
    tbl = aggregate_ads(peers)

    mask = pd.Series(True, index=tbl.index)
    for label, col, op, val in conds:
        s = tbl[col]
        if op == '>':
            m = s > val
        elif op == '<':
            m = s < val
        else:
            m = s.round(2) == round(val, 2)            # "=" compares to 2 decimal places
        mask &= m.fillna(False)                        # ads without this metric never match
    hit = tbl[mask]

    text_desc = ' and '.join(f"{l} {'=' if o == '==' else o} {v:g}" for l, _, o, v in conds)
    if hit.empty:
        return no_update, f"No ads match: {text_desc}"
    return hit['Ad_Name'].tolist(), f"Showing {len(hit)} of {len(tbl)} ads: {text_desc}"

# Register the callback only once
if not any('metric-filter-msg.children' in key for key in app.callback_map):
    @app.callback(
        Output('ad-filter', 'value', allow_duplicate=True),
        Output('metric-filter-msg', 'children'),
        *[Output(f'mf-op-{l}', 'value') for l, _ in METRIC_ROWS],
        *[Output(f'mf-val-{l}', 'value') for l, _ in METRIC_ROWS],
        Input('mf-apply', 'n_clicks'),
        Input('mf-clear', 'n_clicks'),
        *[State(f'mf-op-{l}', 'value') for l, _ in METRIC_ROWS],
        *[State(f'mf-val-{l}', 'value') for l, _ in METRIC_ROWS],
        State('section', 'data'), State('date-range', 'start_date'), State('date-range', 'end_date'),
        prevent_initial_call=True)
    def use_metric_filter(apply_n, clear_n, *rest):
        n = len(METRIC_ROWS)
        ops, vals = rest[:n], rest[n:2 * n]
        section, start, end = rest[2 * n:]
        if ctx.triggered_id == 'mf-clear':
            return [], '', *([None] * n), *([None] * n)          # reset everything
        ads_value, msg = metric_filter_logic(ops, vals, section, start, end)
        return ads_value, msg, *([no_update] * (2 * n))

In [113]:
# ============ STEP 6F: BENCHMARKS ============

# ---------- Settings you can edit: (table column, lowest OK, highest OK). None = no limit ----------
BENCHMARKS = {
    'CPM':   ('CPM',      30,   35),     # RM 30 to RM 35
    'CPC':   ('CPC_Link', 0.5,  1),      # RM 0.50 to RM 1
    'CTR':   ('CTR_Link', 1,    2),      # 1% to 2%
    'CPLPV': ('CPLPV',    None, 1),      # below RM 1
    'CPP':   ('CPP',      None, 70),     # RM 70 or less
    'ROAS':  ('ROAS',     3,    None),   # 3 or higher (higher is better)
}

# ---------- Work out which benchmarks each ad meets ----------
def add_benchmarks(tbl):
    t = tbl.copy()
    met = pd.DataFrame(index=t.index)
    for label, (col, lo, hi) in BENCHMARKS.items():
        s = t[col]
        ok = s.notna()                       # no data = not met
        if lo is not None:
            ok &= s >= lo
        if hi is not None:
            ok &= s <= hi
        met[label] = ok
    n = met.sum(axis=1)
    total = len(BENCHMARKS)
    t['Bench_Met'] = n
    t['Bench_Result'] = [
        'Too early' if sp < MIN_SPEND else ('Meets all' if k == total else f"{k} of {total}")
        for sp, k in zip(t['Spend'], n)
    ]
    return t

# Make every section that uses scored_view get the benchmark columns (safe to re-run)
if '_orig_scored_view' not in globals():
    _orig_scored_view = scored_view

def scored_view(rows, peers):
    view, peers_scored = _orig_scored_view(rows, peers)
    return add_benchmarks(view), peers_scored

# ---------- Add the new columns to the Data Table (only once) ----------
if not any(c['id'] == 'CPLPV' for c in TABLE_COLUMNS):
    i = [c['id'] for c in TABLE_COLUMNS].index('CPC_Link') + 1
    TABLE_COLUMNS[i:i] = [money('CPLPV (RM)', 'CPLPV'), money('CPP (RM)', 'CPP'),
                          money('ROAS', 'ROAS'), text('Benchmark', 'Bench_Result')]

# ---------- Colours: green = met, red = not met ----------
def _bench_styles():
    styles = []
    for label, (col, lo, hi) in BENCHMARKS.items():
        good = ['{%s} is num' % col]
        if lo is not None:
            good.append('{%s} >= %s' % (col, lo))
        if hi is not None:
            good.append('{%s} <= %s' % (col, hi))
        styles.append({'if': {'filter_query': ' && '.join(good), 'column_id': col},
                       'backgroundColor': '#dcfce7', 'color': '#166534'})
        if lo is not None:
            styles.append({'if': {'filter_query': '{%s} is num && {%s} < %s' % (col, col, lo), 'column_id': col},
                           'backgroundColor': '#fee2e2', 'color': '#991b1b'})
        if hi is not None:
            styles.append({'if': {'filter_query': '{%s} is num && {%s} > %s' % (col, col, hi), 'column_id': col},
                           'backgroundColor': '#fee2e2', 'color': '#991b1b'})
    styles += [
        {'if': {'filter_query': '{Bench_Result} = "Meets all"', 'column_id': 'Bench_Result'},
         'backgroundColor': '#dcfce7', 'color': '#166534', 'fontWeight': 'bold'},
        {'if': {'filter_query': '{Bench_Result} = "Too early"', 'column_id': 'Bench_Result'},
         'backgroundColor': '#f3f4f6', 'color': '#6b7280'},
    ]
    return styles

if not any(s.get('if', {}).get('column_id') == 'Bench_Result' for s in TIER_STYLES):
    TIER_STYLES.extend(_bench_styles())

# Step 7: Video Creative Analysis

In [114]:
# ============ STEP 7: VIDEO CREATIVE ANALYSIS ============
from scipy.stats import spearmanr

# ---------- 7A. Settings ----------
MIN_VIDEO_IMPR   = 1000   # video ads with fewer impressions are left out of the charts (too small to trust)
FATIGUE_MIN_FREQ = 1.5    # an ad needs at least this frequency to be flagged for fatigue
MIN_CORR_ADS     = 6      # need at least this many ads to work out a correlation

# ---------- 7B. Small helpers ----------
def video_ads(tbl):
    return tbl[tbl['Video_3s'] > 0]

def eligible(vids):
    return vids[vids['Impressions'] >= MIN_VIDEO_IMPR]

def pct(v):
    return f"{v:.1f}%"

def secs(v):
    return f"{v:.1f}s"

# ---------- 7C. Charts: one bar chart per metric ----------
def fig_video_bar(vids, col, color, average, fmt):
    d = eligible(vids).dropna(subset=[col])
    if d.empty:
        return empty_fig(f"No video ads with at least {MIN_VIDEO_IMPR:,} impressions")
    top = d.sort_values('Impressions', ascending=False).head(TOP_N).sort_values(col, ascending=False)
    fig = go.Figure(go.Bar(x=top[col], y=top['Ad_Name'], orientation='h', marker_color=color,
                           text=[fmt(v) for v in top[col]], textposition='outside',
                           hovertemplate='%{y}<br>%{x:.2f}<extra></extra>'))
    if pd.notna(average):
        fig.add_vline(x=average, line_dash='dash', line_color=GREY,
                      annotation_text=f"Average {fmt(average)}", annotation_position='top')
    fig.update_yaxes(autorange='reversed', automargin=True)
    fig.update_layout(template='plotly_white', height=420, margin=dict(l=10, r=60, t=40, b=30),
                      title=dict(text=f"Top {len(top)} video ads by impressions", font=dict(size=13, color=GREY)))
    return fig

def fig_hook_vs_result(vids):
    d = eligible(vids)
    if d.empty:
        return empty_fig(f"No video ads with at least {MIN_VIDEO_IMPR:,} impressions")
    size = 8 + 30 * d['Spend'] / d['Spend'].max()
    fig = go.Figure(go.Scatter(
        x=d['Hook_Rate'], y=d['Results'], mode='markers', text=d['Ad_Name'], customdata=d['Spend'],
        marker=dict(size=size, color=BLUE, opacity=0.6, line=dict(width=1, color='white')),
        hovertemplate='%{text}<br>Hook rate %{x:.1f}%<br>Results %{y:.0f}<br>Spend RM %{customdata:,.0f}<extra></extra>'))
    fig.update_xaxes(title='Hook rate (%)')
    fig.update_yaxes(title='Results', rangemode='tozero')
    fig.update_layout(template='plotly_white', height=420, margin=dict(l=10, r=10, t=30, b=30))
    return fig

# ---------- 7D. Creative fatigue (rough check) ----------
def fatigue_table(check, ref):
    """check = video ads to test. ref = all video ads in the period (to work out 'normal')."""
    ref = eligible(ref)
    if ref.empty or check.empty:
        return pd.DataFrame(), None
    freq_line = max(FATIGUE_MIN_FREQ, ref['Frequency'].quantile(0.75))
    ctr_med, hook_med = ref['CTR_Link'].median(), ref['Hook_Rate'].median()
    d = eligible(check).copy()
    freq_high = d['Frequency'] >= freq_line
    ctr_weak  = d['CTR_Link'] < ctr_med
    hook_weak = d['Hook_Rate'] < hook_med
    d['Fatigue'] = np.select([freq_high & ctr_weak & hook_weak, freq_high & (ctr_weak | hook_weak)],
                             ['High', 'Possible'], default='OK')
    d['Age_Days'] = (pd.Timestamp(PERIOD_END) - d['Date_Created']).dt.days
    return d, dict(freq_line=freq_line, ctr_med=ctr_med, hook_med=hook_med)

def fatigue_card(fat, lines):
    if lines is None:
        return card('Creative fatigue flags', html.Div('Not enough video data for these filters.', style={'color': GREY}))
    flagged = fat[fat['Fatigue'] != 'OK'].copy()
    flagged['_o'] = flagged['Fatigue'].map({'High': 0, 'Possible': 1})
    flagged = flagged.sort_values(['_o', 'Frequency'], ascending=[True, False]).head(10)
    if flagged.empty:
        body = html.Div("No ad shows fatigue signs.", style={'color': GREEN, 'padding': '10px 0'})
    else:
        shown = flagged[['Ad_Name', 'Frequency', 'CTR_Link', 'Hook_Rate', 'Age_Days', 'Fatigue']].round(2)
        cols = [text('Ad name', 'Ad_Name'), money('Frequency', 'Frequency'), money('CTR (%)', 'CTR_Link'),
                money('Hook rate (%)', 'Hook_Rate'), whole('Age (days)', 'Age_Days'), text('Flag', 'Fatigue')]
        body = dash_table.DataTable(
            columns=cols, data=shown.astype(object).where(shown.notna(), None).to_dict('records'),
            style_cell={'textAlign': 'left', 'padding': '6px', 'fontFamily': 'Arial', 'fontSize': '13px',
                        'whiteSpace': 'normal'},
            style_cell_conditional=[{'if': {'column_id': 'Ad_Name'}, 'minWidth': '160px'}],
            style_header={'fontWeight': 'bold', 'backgroundColor': '#f3f4f6'},
            style_table={'overflowX': 'auto'},
            style_data_conditional=[
                {'if': {'filter_query': '{Fatigue} = "High"', 'column_id': 'Fatigue'},
                 'backgroundColor': '#fee2e2', 'color': '#991b1b', 'fontWeight': 'bold'},
                {'if': {'filter_query': '{Fatigue} = "Possible"', 'column_id': 'Fatigue'},
                 'backgroundColor': '#fef3c7', 'color': '#92400e', 'fontWeight': 'bold'}])
    note = (f"Rough check. High = frequency of {lines['freq_line']:.1f}+ (top quarter of video ads, never under "
            f"{FATIGUE_MIN_FREQ}) AND both CTR and hook rate below the typical ad "
            f"({lines['ctr_med']:.2f}% and {lines['hook_med']:.1f}%). Possible = only one of the two is low. "
            "Your file has one total per ad, so this compares ads with each other. It cannot see a drop over time. "
            "For that, export daily data (Breakdown > Time > Day).")
    return card('Creative fatigue flags', body, note=note)

# ---------- 7E. Correlation matrix: what predicts performance? ----------
CORR_X = [('Hook_Rate', 'Hook rate'), ('Hold_Rate', 'Hold rate'), ('Finish_Rate', 'Finish rate'),
          ('Avg_Play_Sec', 'Avg play time')]
CORR_Y = [('CTR_Link', 'CTR'), ('Results_per_1k', 'Results per 1k impressions'), ('CPP', 'CPP'), ('ROAS', 'ROAS')]

def corr_tables(peer_vids):
    """Spearman correlation (rank based, so a few extreme ads do not dominate)."""
    pool = peer_vids[(peer_vids['Impressions'] >= MIN_VIDEO_IMPR) & (peer_vids['Spend'] >= MIN_SPEND)]
    xl, yl = [l for _, l in CORR_X], [l for _, l in CORR_Y]
    rho = pd.DataFrame(np.nan, index=xl, columns=yl)
    pv, cnt = rho.copy(), rho.copy()
    for xc, xlab in CORR_X:
        for yc, ylab in CORR_Y:
            d = pool[[xc, yc]].dropna()
            cnt.loc[xlab, ylab] = len(d)
            if len(d) >= MIN_CORR_ADS and d[xc].nunique() > 1 and d[yc].nunique() > 1:
                r, p = spearmanr(d[xc], d[yc])
                rho.loc[xlab, ylab], pv.loc[xlab, ylab] = r, p
    return rho, pv, cnt, len(pool)

def fig_corr(rho, pv, cnt, n_pool):
    if n_pool < MIN_CORR_ADS or rho.isna().all().all():
        return empty_fig(f"Need at least {MIN_CORR_ADS} video ads with RM {MIN_SPEND}+ spend "
                         f"and {MIN_VIDEO_IMPR:,}+ impressions (found {n_pool})")
    txt = [[("n/a" if pd.isna(rho.iloc[i, j]) else
             f"{rho.iloc[i, j]:.2f}{'*' if pv.iloc[i, j] < 0.05 else ''}<br>n={cnt.iloc[i, j]:.0f}")
            for j in range(rho.shape[1])] for i in range(rho.shape[0])]
    fig = go.Figure(go.Heatmap(z=rho.values, x=list(rho.columns), y=list(rho.index), text=txt,
                               texttemplate='%{text}', zmin=-1, zmax=1, colorscale='RdBu',
                               colorbar=dict(title='Correlation'), hoverinfo='skip'))
    fig.update_yaxes(autorange='reversed')
    fig.update_layout(template='plotly_white', height=420, margin=dict(l=10, r=10, t=30, b=30))
    return fig

def corr_insight(rho, pv, cnt, n_pool):
    best = None
    for xl in rho.index:
        for yl in ['Results per 1k impressions', 'CPP', 'ROAS']:
            r, p, n = rho.loc[xl, yl], pv.loc[xl, yl], cnt.loc[xl, yl]
            if pd.notna(r) and p < 0.05 and n >= 8 and (best is None or abs(r) > abs(best[2])):
                best = (xl, yl, r, n)
    if best is None:
        return ('info', "No video metric is clearly linked to results, CPP or ROAS yet. "
                        "With so few purchases, patterns are hard to see.")
    xl, yl, r, n = best
    if yl == 'CPP':
        way = "lower CPP (cheaper purchases)" if r < 0 else "higher CPP (more expensive purchases)"
    else:
        way = f"higher {yl}" if r > 0 else f"lower {yl}"
    return ('good' if (r < 0) == (yl == 'CPP') else 'warn',
            f"Across {n:.0f} video ads, {xl.lower()} is the video metric most linked to performance: "
            f"a higher {xl.lower()} goes with {way} (correlation {r:.2f}). This shows a pattern, not proof.")

# ---------- 7F. Insight sentences ----------
def video_insights(vview, peer_vids, fat, corr_line):
    if vview.empty:
        return [('info', 'No video ads for these filters.')]
    out = []
    tv = totals_from_table(vview)
    out.append(('info', f"Across {len(vview)} video ad(s): {pct(tv['Hook_Rate'])} of impressions become a 3-second view "
                        f"(hook rate), {pct(tv['Hold_Rate'])} of those viewers reach ThruPlay (hold rate), and "
                        f"{pct(tv['Finish_Rate'])} watch to the end. Average watch time is {secs(tv['Avg_Play_Sec'])}."
                if pd.notna(tv['Avg_Play_Sec']) else
                f"Across {len(vview)} video ad(s): {pct(tv['Hook_Rate'])} of impressions become a 3-second view "
                f"(hook rate), {pct(tv['Hold_Rate'])} of those viewers reach ThruPlay (hold rate), and "
                f"{pct(tv['Finish_Rate'])} watch to the end."))

    ref = eligible(peer_vids)
    el = eligible(vview)
    if not el.empty and not ref.empty:
        hook_med, hold_med = ref['Hook_Rate'].median(), ref['Hold_Rate'].median()
        best = el.sort_values('Hook_Rate', ascending=False).iloc[0]
        out.append(('good', f"'{best['Ad_Name']}' has the best hook rate: {pct(best['Hook_Rate'])} "
                            f"(typical video ad: {pct(hook_med)})."))
        worst = el.sort_values('Hook_Rate').iloc[0]
        if len(el) > 1 and worst['Hook_Rate'] < hook_med * 0.75:
            out.append(('warn', f"'{worst['Ad_Name']}' has the weakest hook rate: {pct(worst['Hook_Rate'])}. "
                                "The first 3 seconds are not stopping people - test a new opening."))
        a = el[(el['Hook_Rate'] >= hook_med) & (el['Hold_Rate'] < hold_med)].sort_values('Impressions', ascending=False).head(2)
        if len(a):
            names = '; '.join(f"'{r.Ad_Name}' (hook {pct(r.Hook_Rate)}, hold {pct(r.Hold_Rate)})" for r in a.itertuples())
            out.append(('warn', f"Good opening but viewers drop off: {names}. Try shortening the video or reworking the middle."))
        b = el[(el['Hook_Rate'] < hook_med) & (el['Hold_Rate'] >= hold_med)].sort_values('Impressions', ascending=False).head(2)
        if len(b):
            names = '; '.join(f"'{r.Ad_Name}' (hook {pct(r.Hook_Rate)}, hold {pct(r.Hold_Rate)})" for r in b.itertuples())
            out.append(('info', f"Strong content but weak opening: {names}. People who stay like it - improve the first 3 seconds."))

    if fat is not None and not fat.empty:
        flagged = fat[fat['Fatigue'] != 'OK']
        if len(flagged):
            names = ', '.join(f"'{n}'" for n in flagged.sort_values('Frequency', ascending=False)['Ad_Name'].head(3))
            out.append(('warn', f"{len(flagged)} ad(s) show possible creative fatigue (high frequency with weak CTR or hook rate), "
                                f"for example {names}. Consider a fresh version of the creative."))
        else:
            out.append(('good', "No video ad shows creative fatigue signs."))
    if corr_line:
        out.append(corr_line)
    small = len(vview) - len(el)
    if small > 0:
        out.append(('info', f"{small} video ad(s) have under {MIN_VIDEO_IMPR:,} impressions and are left out of the charts."))
    return out

# ---------- 7G. Put the Video page together ----------
def build_video(rows, peers):
    vview = video_ads(aggregate_ads(rows))
    peer_vids = video_ads(aggregate_ads(peers))
    if vview.empty:
        return html.Div(card('Video Creative Analysis', html.Div(
            'No video ads for these filters. Clear the Ad name filter, or pick an ad that has video plays.',
            style={'color': GREY, 'padding': '20px 0'})))

    tv = totals_from_table(vview)
    avg_play = f"{tv['Avg_Play_Sec']:.1f} s" if pd.notna(tv['Avg_Play_Sec']) else '-'
    kpis = html.Div([
        kpi_card('Hook rate', pct(tv['Hook_Rate']), '3-second plays / impressions'),
        kpi_card('Hold rate', pct(tv['Hold_Rate']), 'ThruPlays / 3-second plays'),
        kpi_card('Finish rate', pct(tv['Finish_Rate']), 'Played to 100% / 3-second plays'),
        kpi_card('Avg play time', avg_play, 'Weighted by impressions'),
        kpi_card('Video ads', f"{len(vview)}"),
    ], style={'display': 'grid', 'gridTemplateColumns': 'repeat(auto-fit, minmax(190px, 1fr))',
              'gap': '14px', 'marginBottom': '16px'})

    fat, lines = fatigue_table(vview, peer_vids)
    rho, pv, cnt, n_pool = corr_tables(peer_vids)
    corr_line = corr_insight(rho, pv, cnt, n_pool) if n_pool >= MIN_CORR_ADS else None

    two = {'display': 'grid', 'gridTemplateColumns': 'repeat(auto-fit, minmax(420px, 1fr))',
           'gap': '16px', 'marginBottom': '16px'}
    return html.Div([
        kpis,
        insight_box(video_insights(vview, peer_vids, fat, corr_line)),
        html.Div([card('Hook rate % by ad', graph(fig_video_bar(vview, 'Hook_Rate', BLUE, tv['Hook_Rate'], pct))),
                  card('Hold rate % by ad', graph(fig_video_bar(vview, 'Hold_Rate', GREEN, tv['Hold_Rate'], pct)))], style=two),
        html.Div([card('Finish rate % by ad', graph(fig_video_bar(vview, 'Finish_Rate', AMBER, tv['Finish_Rate'], pct))),
                  card('Average play time by ad', graph(fig_video_bar(vview, 'Avg_Play_Sec', GREY, tv['Avg_Play_Sec'], secs)))], style=two),
        html.Div([card('Video funnel', graph(fig_video_funnel(vview))),
                  card('Hook rate vs result', graph(fig_hook_vs_result(vview)),
                       note='Each dot is one ad. A bigger dot means more spend.')], style=two),
        html.Div([fatigue_card(fat, lines),
                  card('What predicts performance? (correlation matrix)', graph(fig_corr(rho, pv, cnt, n_pool)),
                       note=(f"Based on all {n_pool} video ads in the date period (not affected by the Ad name filter). "
                             "1 = move together, -1 = move opposite, 0 = no link. * = unlikely to be chance. "
                             "For CPP a negative number is good (cheaper purchases). "
                             "n = number of ads behind each number."))], style=two),
    ])

SECTION_BUILDERS['video'] = build_video

# Step 8: Lead Analysis

In [115]:
# ============ STEP 8: LEAD ANALYSIS ============
# ---------- 8A. Settings ----------
N_MATRIX        = 8    # how many ads to show side by side in the "parameters vs ad name" table
MIN_LEADS_JUDGE = 2    # an ad needs at least this many leads before it is called best / worst on CPL
MIN_CLICKS_LEAK = 30   # an ad needs at least this many link clicks before we comment on its click-to-lead rate
FEW_LEADS       = 30   # below this many total leads, show a "numbers are small" warning
WARN_GAP        = 25   # only say "consider pausing" when an ad's CPL is at least this % above average

# ---------- 8B. Lead table ----------
def lead_ads(tbl):
    """Lead ads = ads with at least 1 lead in the chosen dates. Adds two lead-only metrics."""
    lv = tbl[tbl['Leads'] > 0].copy()
    lv['Leads_per_1k'] = ratio(lv['Leads'], lv['Impressions'], 1000)   # "result per impression" x 1,000
    lv['Lead_Rate']    = ratio(lv['Leads'], lv['Link_Clicks'], 100)    # leads per 100 link clicks
    return lv

# ---------- 8C. Charts ----------
def fig_flow(lv):
    d = lv.sort_values('Spend', ascending=False).head(TOP_N)
    stages = [('Hook_Rate', 'Hook rate %'), ('Hold_Rate', 'Hold rate %'),
              ('CTR_Link', 'CTR link click %'), ('Lead_Rate', 'Lead rate % (leads / link clicks)')]
    raw_vals = d[[c for c, _ in stages]].astype(float)
    lo, hi = raw_vals.min(), raw_vals.max()
    z = (raw_vals - lo) / (hi - lo).replace(0, np.nan)          # colour each column against its own best / worst
    z = z.fillna(0.5).where(raw_vals.notna())                   # same value in a column = middle colour
    text_vals = [[("" if pd.isna(v) else f"{v:.1f}%") for v in row] for row in raw_vals.values]
    fig = go.Figure(go.Heatmap(z=z.values, x=[l for _, l in stages], y=d['Ad_Name'], text=text_vals,
                               texttemplate='%{text}', colorscale='RdYlGn', zmin=0, zmax=1, showscale=False,
                               xgap=2, ygap=2, hoverinfo='skip'))
    fig.update_yaxes(autorange='reversed', automargin=True)
    fig.update_xaxes(side='top')
    fig.update_layout(template='plotly_white', height=max(320, 42 * len(d) + 110),
                      margin=dict(l=10, r=10, t=60, b=10))
    return fig

def fig_hook_vs_leads(lv):
    d = lv[(lv['Hook_Rate'].notna()) & (lv['Impressions'] >= MIN_VIDEO_IMPR)]
    if len(d) < 2:
        return empty_fig("Need at least 2 video lead ads to compare hook rate with leads")
    size = 8 + 30 * d['Spend'] / d['Spend'].max()
    fig = go.Figure(go.Scatter(
        x=d['Hook_Rate'], y=d['Leads_per_1k'], mode='markers', text=d['Ad_Name'], customdata=d[['Leads', 'Spend']],
        marker=dict(size=size, color=BLUE, opacity=0.6, line=dict(width=1, color='white')),
        hovertemplate=('%{text}<br>Hook rate %{x:.1f}%<br>Leads per 1k impressions %{y:.2f}'
                       '<br>Leads %{customdata[0]:.0f}<br>Spend RM %{customdata[1]:,.0f}<extra></extra>')))
    fig.update_xaxes(title='Hook rate (%)')
    fig.update_yaxes(title='Leads per 1,000 impressions', rangemode='tozero')
    fig.update_layout(template='plotly_white', height=420, margin=dict(l=10, r=10, t=30, b=30))
    return fig

def fig_leads_cpl(lv, avg_cpl):
    d = lv.sort_values('Leads', ascending=False).head(TOP_N)
    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Bar(x=d['Ad_Name'], y=d['Leads'], name='Leads', marker_color=BLUE), secondary_y=False)
    fig.add_trace(go.Scatter(x=d['Ad_Name'], y=d['CPL'], name='CPL (RM)', mode='lines+markers',
                             line=dict(color=RED)), secondary_y=True)
    if pd.notna(avg_cpl):
        fig.add_trace(go.Scatter(x=d['Ad_Name'], y=[avg_cpl] * len(d), name=f"Average CPL (RM {avg_cpl:,.2f})",
                                 mode='lines', line=dict(color=GREY, dash='dash')), secondary_y=True)
    fig.update_yaxes(title_text='Leads', secondary_y=False)
    fig.update_yaxes(title_text='CPL (RM)', secondary_y=True, showgrid=False, rangemode='tozero')
    fig.update_xaxes(tickangle=-35, automargin=True)
    fig.update_layout(template='plotly_white', height=420, margin=dict(l=10, r=10, t=30, b=10),
                      legend=dict(orientation='h', y=1.12))
    return fig

# ---------- 8D. Data performance table: parameters (rows) vs ad name (columns) ----------
def _money(v): return f"RM {v:,.2f}"
def _whole(v): return f"{v:,.0f}"
def _pct(v):   return f"{v:.2f}%"
def _num2(v):  return f"{v:.2f}"

PARAMS = [   # label, column, formatter, which value is best ('high', 'low' or None)
    ('Amount spent', 'Spend', _money, None), ('Impressions', 'Impressions', _whole, None),
    ('Reach', 'Reach', _whole, None), ('Frequency', 'Frequency', _num2, None),
    ('CPM', 'CPM', _money, 'low'), ('Link clicks', 'Link_Clicks', _whole, 'high'),
    ('CTR link click', 'CTR_Link', _pct, 'high'), ('CPC link click', 'CPC_Link', _money, 'low'),
    ('Hook rate', 'Hook_Rate', _pct, 'high'), ('Hold rate', 'Hold_Rate', _pct, 'high'),
    ('Leads', 'Leads', _whole, 'high'), ('CPL', 'CPL', _money, 'low'),
    ('Leads per 1k impressions', 'Leads_per_1k', _num2, 'high'),
    ('Lead rate (leads / link clicks)', 'Lead_Rate', _pct, 'high'),
]

def params_table(lv):
    top = lv.sort_values('Spend', ascending=False).head(N_MATRIX).reset_index(drop=True)
    cols = [{'name': 'Parameter', 'id': 'param'}] + [{'name': n, 'id': f'a{i}'} for i, n in enumerate(top['Ad_Name'])]
    data, best_cells = [], []
    for r, (label, col, fmt, better) in enumerate(PARAMS):
        row = {'param': label}
        for i, v in enumerate(top[col]):
            row[f'a{i}'] = '-' if pd.isna(v) else fmt(v)
        data.append(row)
        vals = top[col].dropna()
        if better and len(vals) >= 2 and vals.nunique() > 1:
            best_i = vals.idxmax() if better == 'high' else vals.idxmin()
            best_cells.append({'if': {'row_index': r, 'column_id': f'a{best_i}'},
                               'backgroundColor': '#dcfce7', 'fontWeight': 'bold'})
    table = dash_table.DataTable(
        columns=cols, data=data, style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'right', 'padding': '8px', 'fontFamily': 'Arial', 'fontSize': '13px',
                    'minWidth': '120px', 'whiteSpace': 'normal'},
        style_cell_conditional=[{'if': {'column_id': 'param'}, 'textAlign': 'left', 'fontWeight': 'bold',
                                 'minWidth': '190px'}],
        style_header={'fontWeight': 'bold', 'backgroundColor': '#f3f4f6', 'whiteSpace': 'normal', 'height': 'auto'},
        style_data_conditional=best_cells)
    more = f" Showing the top {len(top)} of {len(lv)} lead ads by spend." if len(lv) > len(top) else ""
    return table, "Green = best ad for that row (highest, or lowest for costs)." + more

# ---------- 8E. Summary table ----------
SUMMARY_COLUMNS = [text('Ad name', 'Ad_Name'), money('Hook rate (%)', 'Hook_Rate'), money('Hold rate (%)', 'Hold_Rate'),
                   whole('Link clicks', 'Link_Clicks'), money('CTR link click (%)', 'CTR_Link'),
                   money('CPC link click (RM)', 'CPC_Link'), money('Leads per 1k impressions', 'Leads_per_1k'),
                   whole('Leads', 'Leads'), money('CPL (RM)', 'CPL'), whole('Winning ad score', 'Win_Score'),
                   text('Ad tier', 'Ad_Tier'), text('Why this tier', 'Tier_Reason')]

def summary_table(lv):
    d = lv.sort_values('Win_Score', ascending=False)
    shown = d[[c['id'] for c in SUMMARY_COLUMNS]].round(2)
    return dash_table.DataTable(
        columns=SUMMARY_COLUMNS, data=shown.astype(object).where(shown.notna(), None).to_dict('records'),
        page_size=10, sort_action='native', export_format='csv', style_table={'overflowX': 'auto'},
        style_cell={'textAlign': 'left', 'padding': '8px', 'fontFamily': 'Arial', 'fontSize': '13px',
                    'minWidth': '90px', 'maxWidth': '280px', 'whiteSpace': 'normal'},
        style_cell_conditional=[{'if': {'column_id': 'Ad_Name'}, 'minWidth': '200px'},
                                {'if': {'column_id': 'Tier_Reason'}, 'minWidth': '230px'}],
        style_header={'fontWeight': 'bold', 'backgroundColor': '#f3f4f6'}, style_data_conditional=TIER_STYLES)

# ---------- 8F. Insight sentences ----------
def lead_insights(lv, peer_lv, campaign_spend, peer_campaign_spend):
    tot = totals_from_table(lv)
    ptot = totals_from_table(peer_lv)
    tot['CPL']  = campaign_spend / tot['Leads'] if tot['Leads'] > 0 else float('nan')
    ptot['CPL'] = peer_campaign_spend / ptot['Leads'] if ptot['Leads'] > 0 else float('nan')
    bench = ptot['CPL']
    out = [('info', f"{len(lv)} lead ad(s) got {tot['Leads']:,.0f} leads for {rm(tot['Spend'])}: "
                    f"CPL {rm(tot['CPL'])}, or {tot['Leads'] / tot['Impressions'] * 1000:.2f} leads per 1,000 impressions.")]

    judged = lv[(lv['Spend'] >= MIN_SPEND) & (lv['Leads'] >= MIN_LEADS_JUDGE)]
    if len(judged) == 1 and pd.notna(bench):
        a = judged.iloc[0]
        out.append(('info', f"'{a['Ad_Name']}' has a CPL of {rm(a['CPL'])}, {vs_bench(a['CPL'], bench)} "
                            f"the average CPL of {rm(bench)} for all lead ads in these dates."))
    if len(judged) >= 2 and pd.notna(bench):
        best = judged.sort_values('CPL').iloc[0]
        s = (f"'{best['Ad_Name']}' has the lowest CPL: {rm(best['CPL'])}, {vs_bench(best['CPL'], bench)} "
             f"the average CPL of {rm(bench)}, from {best['Leads']:.0f} leads")
        if best['Leads'] < 5:
            s += " (few leads, so treat it with care)"
        out.append(('good' if best['CPL'] <= bench else 'info', s + '.'))
        worst = judged.sort_values('CPL').iloc[-1]
        gap = (worst['CPL'] - bench) / bench * 100
        if gap >= WARN_GAP:
            out.append(('warn', f"'{worst['Ad_Name']}' has the highest CPL: {rm(worst['CPL'])}, "
                                f"{vs_bench(worst['CPL'], bench)} the average - consider pausing or fixing it."))
        elif worst['Ad_Name'] != best['Ad_Name']:
            out.append(('info', f"'{worst['Ad_Name']}' has the highest CPL: {rm(worst['CPL'])}, "
                                f"{vs_bench(worst['CPL'], bench)} the average, which is not far off."))

    # where do people drop out? (only ads with enough clicks)
    ref = peer_lv[(peer_lv['Link_Clicks'] >= MIN_CLICKS_LEAK) & (peer_lv['Lead_Rate'] <= 100)]
    ok = lv[(lv['Link_Clicks'] >= MIN_CLICKS_LEAK) & (lv['Lead_Rate'] <= 100)]
    if len(ref) >= 3 and not ok.empty:
        ctr_med, lr_med = ref['CTR_Link'].median(), ref['Lead_Rate'].median()
        a = ok[(ok['CTR_Link'] >= ctr_med) & (ok['Lead_Rate'] < lr_med * 0.5)].sort_values('Link_Clicks', ascending=False).head(2)
        if len(a):
            names = '; '.join(f"'{r.Ad_Name}' (CTR {r.CTR_Link:.2f}%, {r.Lead_Rate:.1f} leads per 100 clicks)" for r in a.itertuples())
            out.append(('warn', f"People click but few become leads: {names}. Typical is {lr_med:.1f} per 100 clicks, "
                                "so check the form or landing page."))
        b = ok[(ok['Lead_Rate'] >= lr_med) & (ok['CTR_Link'] < ctr_med * 0.75)].sort_values('Lead_Rate', ascending=False).head(2)
        if len(b):
            names = '; '.join(f"'{r.Ad_Name}' (CTR {r.CTR_Link:.2f}%, {r.Lead_Rate:.1f} leads per 100 clicks)" for r in b.itertuples())
            out.append(('info', f"Clicks turn into leads well but too few people click: {names}. "
                                "Try a stronger creative or call-to-action."))

    # does hook rate predict leads? (video lead ads only)
    vid = peer_lv[(peer_lv['Hook_Rate'].notna()) & (peer_lv['Impressions'] >= MIN_VIDEO_IMPR)]
    if len(vid) >= MIN_CORR_ADS and vid['Hook_Rate'].nunique() > 1 and vid['Leads_per_1k'].nunique() > 1:
        r, p = spearmanr(vid['Hook_Rate'], vid['Leads_per_1k'])
        if p < 0.05:
            way = "more" if r > 0 else "fewer"
            out.append(('good' if r > 0 else 'warn',
                        f"Across {len(vid)} video lead ads, a higher hook rate goes with {way} leads per 1,000 "
                        f"impressions (correlation {r:.2f}). This shows a pattern, not proof."))
        else:
            out.append(('info', f"Across {len(vid)} video lead ads, hook rate is not clearly linked to leads per 1,000 impressions."))

    sc = lv[lv['Ad_Tier'] == 'Scale'].sort_values('Win_Score', ascending=False).head(2)
    if len(sc):
        out.append(('good', "Tagged Scale: " + '; '.join(f"'{r.Ad_Name}' ({r.Tier_Reason})" for r in sc.itertuples()) + "."))
    kl = lv[lv['Ad_Tier'] == 'Kill']
    if len(kl):
        out.append(('warn', f"{len(kl)} lead ad(s) are tagged Kill, together RM {kl['Spend'].sum():,.2f} of spend."))
    if tot['Leads'] < FEW_LEADS:
        out.append(('info', f"Only {tot['Leads']:.0f} leads in total, so one more lead can change an ad's CPL a lot. "
                            "Treat the rankings as a guide."))
    return out

# ---------- 8G. Put the Lead page together ----------
def build_lead(rows, peers):
    view, _ = scored_view(rows, peers)
    lv = lead_ads(view)
    peer_lv = lead_ads(aggregate_ads(peers))
    campaign_spend = totals_from_table(view)['Spend']         # spend on ALL ads, not just lead ads
    peer_campaign_spend = totals(peers)['Spend']
    if lv.empty:
        hint = (f"{len(peer_lv)} ad(s) have leads in the chosen dates. Try clearing the Ad name filter."
                if len(peer_lv) else "No ad has any leads in the chosen dates.")
        return html.Div(card('Lead Analysis', html.Div(
            ["No leads for these filters. An ad shows here once it has at least 1 lead. ", hint],
            style={'color': GREY, 'padding': '20px 0'})))

    tot = totals_from_table(lv)
    tot['CPL'] = campaign_spend / tot['Leads'] if tot['Leads'] > 0 else float('nan')
    kpis = html.Div([
        kpi_card('Number of leads', f"{tot['Leads']:,.0f}"),
        kpi_card('CPL', rm(tot['CPL']), 'Total campaign spend / leads'),
        kpi_card('Lead ads', f"{len(lv)}", 'Ads with at least 1 lead'),
        kpi_card('Spend on lead ads', rm(tot['Spend'])),
        kpi_card('Leads per 1k impressions', f"{tot['Leads'] / tot['Impressions'] * 1000:.2f}"),
    ], style={'display': 'grid', 'gridTemplateColumns': 'repeat(auto-fit, minmax(190px, 1fr))',
              'gap': '14px', 'marginBottom': '16px'})

    ptable, pnote = params_table(lv)
    two = {'display': 'grid', 'gridTemplateColumns': 'repeat(auto-fit, minmax(420px, 1fr))',
           'gap': '16px', 'marginBottom': '16px'}
    one = {'marginBottom': '16px'}
    return html.Div([
        kpis,
        insight_box(lead_insights(lv, peer_lv, campaign_spend, peer_campaign_spend)),
        html.Div([card('Flow funnel by ad: hook > hold > click > lead', graph(fig_flow(lv)),
                       note='Read left to right. Each column is coloured against the best and worst ad in that column '
                            '(green = best). Blank = not applicable, for example image ads have no hook or hold rate.'),
                  card('Hook rate vs results per 1k impressions', graph(fig_hook_vs_leads(lv)),
                       note='Results = leads. Each dot is one video lead ad. A bigger dot means more spend.')], style=two),
        html.Div(card('Leads and CPL by ad', graph(fig_leads_cpl(lv, tot['CPL']))), style=one),
        html.Div(card('Data performance table (parameters vs ad name)', ptable, note=pnote), style=one),
        html.Div(card('Summary table', summary_table(lv),
                      note="Leads per 1k impressions = 'result per impression' x 1,000. Score and tier work as in the Data Table."),
                 style=one),
    ])

SECTION_BUILDERS['lead'] = build_lead

In [116]:
   p = filter_rows(clean, PERIOD_START, PERIOD_END)
   l = aggregate_ads(p); l = l[l['Leads'] > 0]
   total_period_spend = p['Spend'].sum()   # spend on ALL ads in the period, not just lead ads
   print(len(l), "lead ads |", l['Leads'].sum(), "leads | CPL", round(total_period_spend / l['Leads'].sum(), 2))

12 lead ads | 25.0 leads | CPL 591.79


# Step 9: Conversion Analysis

In [117]:
# ============ STEP 9 (REVISED): CONVERSION ANALYSIS ============
import math
from scipy.stats import fisher_exact, norm, nbinom

# ---------- 9A. Settings ----------
MIN_CLICKS_CVR = 30     # an ad needs this many link clicks before its LPV rate counts
MIN_IMPR_CTR   = 2000   # an ad needs this many impressions before its (unique) CTR counts
CONFIDENT_BUYS = 3      # below this many purchases, say "few purchases" / "wait for more data"
FEW_PURCHASES  = 30     # below this many total purchases, show a "numbers are small" warning
SIG_LEVEL      = 0.05   # "is the gap real?" test: below this p-value we say yes
EFFICIENCY = {           # what-if: how much does cost per purchase rise when the budget grows?
    'flat':   ('Cost per purchase stays the same', 1.0),
    'mild':   ('Cost rises a little as budget grows', 0.9),
    'strong': ('Cost rises a lot as budget grows', 0.75),
}
WHATIF_KEY = 'mild'      # the what-if simulator always uses this one (factor 0.9)

# ---------- 9B. Data for this section ----------
def conv_tables(rows, peers):
    """Ad tables with the Step 6 score/tier, plus a clean LPV rate. Scores compare against ALL ads in the date range."""
    view, peers_tbl = scored_view(rows, peers)
    for t in (view, peers_tbl):
        t['LPV_Clean'] = t['LPV_Rate'].where(t['Landing_Page_Views'] > 0)    # no LPV data = not applicable
    return view, peers_tbl

def overall_unique_ctr(v):
    return v['Unique_Link_Clicks'].sum() / v['Reach'].sum() * 100 if v['Reach'].sum() > 0 else np.nan

def overall_lpv(v):
    h = v[v['Landing_Page_Views'] > 0]
    return h['Landing_Page_Views'].sum() / h['Link_Clicks'].sum() * 100 if h['Link_Clicks'].sum() > 0 else np.nan

def f2(v, unit=''):
    return '-' if pd.isna(v) else f"{v:.2f}{unit}"

def num(v):
    return f"{v:,.1f}" if abs(v) < 10 else f"{v:,.0f}"

def plural(n, word):
    return f"{n:,.0f} {word}" + ("" if round(n) == 1 else "s")

def grey_note(msg):
    return html.Div(msg, style={'color': GREY, 'padding': '10px 0'})

# ---------- 9C. Insight sentences (CPP, CTR, LPV) ----------
def conversion_insights(view, peers_tbl, tot):
    out = []
    ptot = totals_from_table(peers_tbl)
    bench = ptot['CPP']

    # ---- CPP ----
    if tot['Purchases'] > 0:
        aov, cpp = tot['Purchase_Value'] / tot['Purchases'], tot['CPP']
        if cpp > aov:
            out.append(('warn', f"CPP: each purchase costs {rm(cpp)} in ads and brings in {rm(aov)} on average, so you lose "
                                f"about {rm(cpp - aov)} per purchase before other costs (ROAS {tot['ROAS']:.2f})."))
        else:
            out.append(('good', f"CPP: each purchase costs {rm(cpp)} in ads and brings in {rm(aov)} on average, so you keep "
                                f"about {rm(aov - cpp)} per purchase before other costs (ROAS {tot['ROAS']:.2f})."))
    else:
        out.append(('warn', "CPP: there are no purchases for these filters yet."))
    judged = view[(view['Spend'] >= MIN_SPEND) & (view['Purchases'] >= 1)]
    if len(judged) == 1 and pd.notna(bench):
        a = judged.iloc[0]
        out.append(('info', f"'{a['Ad_Name']}' has a CPP of {rm(a['CPP'])}, {vs_bench(a['CPP'], bench)} the average CPP of "
                            f"{rm(bench)} for all ads in these dates."))
    if len(judged) >= 2 and pd.notna(bench):
        best = judged.sort_values('CPP').iloc[0]
        s = (f"'{best['Ad_Name']}' has the lowest CPP: {rm(best['CPP'])}, {vs_bench(best['CPP'], bench)} the average CPP of "
             f"{rm(bench)}, from {best['Purchases']:.0f} purchase(s)")
        if best['Purchases'] < CONFIDENT_BUYS:
            s += " (few purchases, so treat it with care)"
        out.append(('good' if best['CPP'] <= bench else 'info', s + '.'))
        worst = judged.sort_values('CPP').iloc[-1]
        if (worst['CPP'] - bench) / bench * 100 >= WARN_GAP:
            s = f"'{worst['Ad_Name']}' has the highest CPP: {rm(worst['CPP'])}, {vs_bench(worst['CPP'], bench)} the average"
            s += (f", but it has only {worst['Purchases']:.0f} purchase(s) so far - wait for more data before pausing it."
                  if worst['Purchases'] < CONFIDENT_BUYS else " - consider pausing or fixing it.")
            out.append(('warn', s))
    no_buy = view[(view['Spend'] >= MIN_SPEND) & (view['Purchases'] == 0)]
    if len(no_buy) > 0 and tot['Spend'] > 0:
        out.append(('info', f"{len(no_buy)} ad(s) spent {rm(no_buy['Spend'].sum())} ({no_buy['Spend'].sum() / tot['Spend'] * 100:.0f}% "
                            "of spend) without a purchase yet. Not all of them have had a fair chance - see the Ad tier column."))

    # ---- CTR ----
    uctr = overall_unique_ctr(view)
    line = f"CTR: {tot['CTR_Link']:.2f}% of impressions become a link click"
    if pd.notna(uctr):
        line += f", and {uctr:.2f}% of the people reached clicked (unique CTR)"
    el = view[view['Impressions'] >= MIN_IMPR_CTR]
    if len(el) >= 2:
        b, w = el.sort_values('CTR_Link').iloc[-1], el.sort_values('CTR_Link').iloc[0]
        line += f". Best: '{b['Ad_Name']}' ({b['CTR_Link']:.2f}%)"
        if w['CTR_Link'] < tot['CTR_Link'] * 0.6:
            line += (f". Weakest: '{w['Ad_Name']}' ({w['CTR_Link']:.2f}%) - people see it but do not click, "
                     "so test a new creative or offer")
    out.append(('info', line + '.'))

    # ---- LPV ----
    lpv = overall_lpv(view)
    if pd.notna(lpv):
        line = f"LPV: {lpv:.0f}% of link clicks reach the landing page, so {100 - lpv:.0f}% click but leave before it loads"
        el = view[(view['Landing_Page_Views'] > 0) & (view['Link_Clicks'] >= MIN_CLICKS_CVR)]
        if len(el) >= 2:
            w = el.sort_values('LPV_Clean').iloc[0]
            if w['LPV_Clean'] < lpv - 10:
                line += (f". Worst: '{w['Ad_Name']}' (only {w['LPV_Clean']:.0f}%) - check page speed, "
                         "or whether the ad attracts accidental clicks")
        out.append(('warn' if lpv < 70 else 'info', line + '.'))
    if tot['Link_Clicks'] > 0 and tot['Purchases'] > 0:
        out.append(('info', f"About 1 in {tot['Link_Clicks'] / tot['Purchases']:.0f} link clicks becomes a purchase."))

    sc = view[view['Ad_Tier'] == 'Scale'].sort_values('Win_Score', ascending=False).head(2)
    if len(sc):
        out.append(('good', "Tagged Scale: " + '; '.join(f"'{r.Ad_Name}' ({r.Tier_Reason})" for r in sc.itertuples()) + "."))
    if tot['Purchases'] < FEW_PURCHASES:
        out.append(('info', f"Only {tot['Purchases']:.0f} purchases in total, so one more purchase can change an ad's CPP a lot. "
                            "Treat rankings as a guide."))
    return out

# ---------- 9D. Bar chart you can switch between CPP / unique CTR / LPV rate ----------
def fig_conv_switch(view):
    specs = [
        dict(key='CPP', title='CPP (RM)', col='CPP', asc=True, good_low=True, suffix='', fmt=lambda v: f"RM {v:,.0f}",
             mask=(view['Purchases'] >= 1) & (view['Spend'] >= MIN_SPEND),
             avg=view['Spend'].sum() / view['Purchases'].sum() if view['Purchases'].sum() > 0 else np.nan),
        dict(key='Unique CTR', title='Unique CTR (%)', col='Unique_CTR', asc=False, good_low=False, suffix='%',
             fmt=lambda v: f"{v:.2f}%", mask=(view['Impressions'] >= MIN_IMPR_CTR) & (view['Reach'] > 0),
             avg=overall_unique_ctr(view)),
        dict(key='LPV rate', title='LPV rate (%)', col='LPV_Clean', asc=False, good_low=False, suffix='%',
             fmt=lambda v: f"{v:.0f}%", mask=(view['Landing_Page_Views'] > 0) & (view['Link_Clicks'] >= MIN_CLICKS_CVR),
             avg=overall_lpv(view)),
    ]
    data = []
    for s in specs:
        d = view[s['mask']].dropna(subset=[s['col']]).sort_values('Spend', ascending=False).head(TOP_N)
        s['d'] = d.sort_values(s['col'], ascending=s['asc'])
        data.append(s)
    if all(s['d'].empty for s in data):
        return empty_fig('Not enough data for these filters')

    fig = go.Figure()
    for s in data:
        d = s['d']
        better = (d[s['col']] <= s['avg']) if s['good_low'] else (d[s['col']] >= s['avg'])
        fig.add_trace(go.Bar(x=d['Ad_Name'], y=d[s['col']], name=s['key'], visible=False,
                             marker_color=[GREEN if b else AMBER for b in better],
                             text=[s['fmt'](v) for v in d[s['col']]], textposition='outside',
                             hovertemplate='%{x}<br>%{y:.2f}<extra></extra>'))
        fig.add_trace(go.Scatter(x=d['Ad_Name'], y=[s['avg']] * len(d), mode='lines', visible=False,
                                 name=f"Average ({s['fmt'](s['avg']) if pd.notna(s['avg']) else '-'})",
                                 line=dict(color=GREY, dash='dash')))
    def empty_note(s):
        return [] if len(s['d']) else [dict(text=f"No ads qualify for {s['key']} with these filters",
                                            x=0.5, y=0.5, xref='paper', yref='paper', showarrow=False,
                                            font=dict(size=14, color=GREY))]
    buttons = []
    for i, s in enumerate(data):
        vis = [False] * (2 * len(data))
        vis[2 * i] = vis[2 * i + 1] = True
        buttons.append(dict(label=s['key'], method='update',
                            args=[{'visible': vis},
                                  {'xaxis.categoryorder': 'array', 'xaxis.categoryarray': list(s['d']['Ad_Name']),
                                   'yaxis.title.text': s['title'], 'yaxis.ticksuffix': s['suffix'],
                                   'annotations': empty_note(s)}]))
    first = data[0]
    for t in (fig.data[0], fig.data[1]):
        t.visible = True
    fig.update_layout(template='plotly_white', height=460, margin=dict(l=10, r=10, t=70, b=10),
                      xaxis=dict(categoryorder='array', categoryarray=list(first['d']['Ad_Name']),
                                 tickangle=-35, automargin=True),
                      yaxis=dict(title=first['title'], ticksuffix=first['suffix']),
                      annotations=empty_note(first), legend=dict(orientation='h', y=1.02, x=0.35),
                      updatemenus=[dict(type='buttons', direction='right', x=0, y=1.13, xanchor='left',
                                        showactive=True, buttons=buttons)])
    return fig

# ---------- 9E. Compare two ads: is the gap real? ----------
CMP_METRICS = {
    'CPP': 'Cost per purchase',
    'CTR': 'Click-through rate (link clicks / impressions)',
    'LPV': 'Landing page view rate (landing page views / link clicks)',
    'CVR': 'Purchase rate (purchases / landing page views)',
}
CMP_PARTS = {'CTR': ('Link_Clicks', 'Impressions', 'link clicks', 'impressions'),
             'LPV': ('Landing_Page_Views', 'Link_Clicks', 'landing page views', 'link clicks'),
             'CVR': ('Purchases', 'Landing_Page_Views', 'purchases', 'landing page views')}

def two_prop_p(x1, n1, x2, n2):
    """p-value for 'are these two rates really different?'. Exact test when counts are small."""
    if min(x1, x2, n1 - x1, n2 - x2) < 10:
        return fisher_exact([[x1, n1 - x1], [x2, n2 - x2]])[1]
    pp = (x1 + x2) / (n1 + n2)
    se = math.sqrt(pp * (1 - pp) * (1 / n1 + 1 / n2))
    return 1.0 if se == 0 else 2 * norm.sf(abs((x1 / n1 - x2 / n2) / se))

def compare_ads(a, b, key):
    """Returns (result dict, None) or (None, message)."""
    if key == 'CPP':
        p1, p2, s1, s2 = int(a['Purchases']), int(b['Purchases']), float(a['Spend']), float(b['Spend'])
        if p1 + p2 == 0:
            return None, "Neither ad has a purchase yet, so there is nothing to compare."
        from scipy.stats import binomtest
        pval = binomtest(p1, p1 + p2, s1 / (s1 + s2)).pvalue    # purchases should split in line with spend if equal
        show = lambda v, p: rm(v) if p else 'no purchase yet'
        r1, r2 = p1 / s1, p2 / s2                                # purchases per RM (higher = cheaper purchases)
        gap = ((s1 / p1) / (s2 / p2) - 1) * 100 if p1 and p2 else None
        return dict(a=(show(a['CPP'], p1), f"{plural(p1, 'purchase')} from {rm(s1)}"),
                    b=(show(b['CPP'], p2), f"{plural(p2, 'purchase')} from {rm(s2)}"),
                    p=pval, winner=None if r1 == r2 else ('a' if r1 > r2 else 'b'), gap=gap,
                    low_is_good=True, events=min(p1, p2), event_name='purchases'), None
    xc, nc, xname, nname = CMP_PARTS[key]
    x1, n1, x2, n2 = float(a[xc]), float(a[nc]), float(b[xc]), float(b[nc])
    if n1 <= 0 or n2 <= 0:
        return None, f"One of the ads has no {nname}, so this rate cannot be worked out."
    if x1 > n1 or x2 > n2:
        return None, f"The export has more {xname} than {nname} for one ad, so this rate cannot be trusted."
    x1, n1, x2, n2 = int(round(x1)), int(round(n1)), int(round(x2)), int(round(n2))
    r1, r2 = x1 / n1, x2 / n2
    gap = (r1 / r2 - 1) * 100 if r2 > 0 else None
    return dict(a=(f"{r1 * 100:.2f}%", f"{x1:,} of {n1:,} {nname}"),
                b=(f"{r2 * 100:.2f}%", f"{x2:,} of {n2:,} {nname}"),
                p=two_prop_p(x1, n1, x2, n2), winner=None if r1 == r2 else ('a' if r1 > r2 else 'b'), gap=gap,
                low_is_good=False, events=min(x1, x2), event_name=xname), None

def compare_render(name_a, name_b, res):
    win = res['winner']
    pval = res['p']
    def box(tag, name, val, detail):
        return html.Div([html.Div(name, style={'fontWeight': 'bold', 'color': NAVY, 'fontSize': '13px'}),
                         html.Div(val, style={'fontSize': '24px', 'fontWeight': 'bold', 'color': NAVY, 'margin': '4px 0'}),
                         html.Div(detail, style={'fontSize': '12px', 'color': GREY})],
                        style={'padding': '12px', 'borderRadius': '8px', 'backgroundColor': '#f9fafb',
                               'border': f"2px solid {GREEN if win == tag else '#e5e7eb'}"})
    boxes = html.Div([box('a', name_a, *res['a']), box('b', name_b, *res['b'])],
                     style={'display': 'grid', 'gridTemplateColumns': '1fr 1fr', 'gap': '10px', 'marginBottom': '10px'})
    if pval < 0.01:
        chance = "less than once in 100 times"
    elif pval >= 0.95:
        chance = "almost every time"
    else:
        chance = f"about {max(1, round(pval * 100))} times in 100"
    if win is None:
        level, line = 'info', "The two ads have exactly the same number, so there is no gap to test."
    else:
        wn = name_a if win == 'a' else name_b
        gap = ''
        if res['gap'] is not None:
            g = abs(res['gap'])
            gap = f" ({name_a} is {g:.0f}% {'higher' if res['gap'] >= 0 else 'lower'} than {name_b})"
        if pval < SIG_LEVEL:
            level, line = 'good', (f"Yes, the gap is probably real{gap}. If the two ads were truly equal, a gap this big "
                                   f"would show up by luck {chance}. '{wn}' looks better.")
        elif pval < 0.20:
            level, line = 'warn', (f"Not clear yet{gap}. '{wn}' looks better, but a gap this big shows up by luck "
                                   f"{chance}. Wait for more data before choosing.")
        else:
            level, line = 'info', (f"No, the gap could easily be luck{gap}. A gap this big shows up by luck {chance}. "
                                   "Treat the two ads as equal for now.")
    kids = [boxes, html.Div([html.Span('\u25cf ', style={'color': {'good': GREEN, 'warn': AMBER, 'info': BLUE}[level]}), line])]
    if res['events'] < 5:
        kids.append(html.Div(f"One ad has fewer than 5 {res['event_name']}, so even a big gap is easy to get by luck.",
                             style={'fontSize': '12px', 'color': GREY, 'marginTop': '6px'}))
    return html.Div(kids)

def compare_from(peer_tbl, name_a, name_b, key):
    if not name_a or not name_b:
        return grey_note('Pick two ads.')
    if name_a == name_b:
        return grey_note('Pick two different ads.')
    ra, rb = peer_tbl[peer_tbl['Ad_Name'] == name_a], peer_tbl[peer_tbl['Ad_Name'] == name_b]
    if ra.empty or rb.empty:
        return grey_note('One of these ads has no data in the chosen dates.')
    res, msg = compare_ads(ra.iloc[0], rb.iloc[0], key)
    return grey_note(msg) if res is None else compare_render(name_a, name_b, res)

def compare_logic(name_a, name_b, key, start, end):
    _, peers = rows_for('conversion', start, end, None)      # not affected by the Ad name filter
    peer_tbl = aggregate_ads(peers)
    return compare_from(peer_tbl, name_a, name_b, key)

# ---------- 9F. Budget what-if simulator (one ad at a time) ----------
def slider_setup(S):
    """Slider runs from 0 to about 3x the current spend, in round steps."""
    step = max(1, 10 ** int(math.floor(math.log10(max(S * 3, 10) / 60))))
    max_b = int(math.ceil(S * 3 / step) * step)
    return max_b, step, int(round(S))

def whatif_render(tot, budget, eff_key):
    n, S = tot['Purchases'], tot['Spend']
    if n < 1 or S <= 0:
        return grey_note('No purchases yet for this ad, so the simulator has nothing to work from.'), empty_fig('No purchases yet')
    beta = EFFICIENCY[eff_key][1]
    max_b, step, default = slider_setup(S)
    B = float(default if budget is None else budget)
    E = n * (B / S) ** beta if B > 0 else 0.0
    if E > 0:
        lo, hi = nbinom.ppf([0.1, 0.9], n, n / (n + E))
    else:
        lo = hi = 0.0
    aov = tot['Purchase_Value'] / n if tot['Purchase_Value'] > 0 else np.nan

    change = (B / S - 1) * 100
    rows = [('Budget', rm(S), f"{rm(B)} ({change:+.0f}%)"),
            ('Purchases', f"{n:,.0f}", f"about {num(E)} ({lo:.0f} to {hi:.0f})" if E > 0 else '0'),
            ('Cost per purchase (CPP)', rm(S / n), rm(B / E) if E > 0 else '-')]
    if pd.notna(aov):
        rows += [('Purchase value', rm(tot['Purchase_Value']), rm(E * aov)),
                 ('ROAS', f"{tot['Purchase_Value'] / S:.2f}", f"{E * aov / B:.2f}" if B > 0 else '-')]
    cell = {'padding': '6px 10px', 'borderBottom': '1px solid #e5e7eb', 'textAlign': 'right'}
    table = html.Table([
        html.Thead(html.Tr([html.Th('', style=cell), html.Th('Now', style=cell), html.Th('What-if', style=cell)])),
        html.Tbody([html.Tr([html.Td(a, style={**cell, 'textAlign': 'left', 'fontWeight': 'bold'}),
                             html.Td(b, style=cell), html.Td(c, style={**cell, 'fontWeight': 'bold', 'color': NAVY})])
                    for a, b, c in rows])],
        style={'borderCollapse': 'collapse', 'width': '100%', 'fontSize': '14px'})
    kids = [table]
    if n < 10:
        kids.append(html.Div(f"Based on only {plural(n, 'purchase')}, so the range is wide.",
                             style={'marginTop': '8px', 'color': AMBER, 'fontSize': '13px'}))

    xs = np.linspace(max_b / 60, max_b, 60)
    Ex = n * (xs / S) ** beta
    lo_x, hi_x = nbinom.ppf(0.1, n, n / (n + Ex)), nbinom.ppf(0.9, n, n / (n + Ex))
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=xs, y=hi_x, mode='lines', line=dict(width=0), showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=xs, y=lo_x, mode='lines', line=dict(width=0), fill='tonexty',
                             fillcolor='rgba(59,130,246,0.18)', name='Likely range (8 in 10)', hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=xs, y=Ex, mode='lines', line=dict(color=BLUE), name='Expected purchases',
                             hovertemplate='RM %{x:,.0f}<br>%{y:.1f} purchases<extra></extra>'))
    fig.add_trace(go.Scatter(x=[S], y=[n], mode='markers', marker=dict(size=11, color=GREY), name='Now',
                             hovertemplate='Now: RM %{x:,.0f}, %{y:.0f} purchases<extra></extra>'))
    fig.add_trace(go.Scatter(x=[B], y=[E], mode='markers', marker=dict(size=13, color=RED, symbol='diamond'),
                             name='What-if', hovertemplate='What-if: RM %{x:,.0f}, %{y:.1f} purchases<extra></extra>'))
    fig.update_xaxes(title='Budget (RM)', range=[0, max_b])
    fig.update_yaxes(title='Purchases', rangemode='tozero')
    fig.update_layout(template='plotly_white', height=380, margin=dict(l=10, r=10, t=30, b=30),
                      legend=dict(orientation='h', y=1.15))
    return html.Div(kids), fig

def ad_totals(ad, start, end):
    """Totals for ONE ad in the chosen date range (not affected by the Ad name filter)."""
    _, peers = rows_for('conversion', start, end, None)
    return totals(peers[peers['Ad_Name'] == ad])

def whatif_logic(budget, ad, start, end):
    return whatif_render(ad_totals(ad, start, end), budget, WHATIF_KEY)

# ---------- 9G. Pick sensible default ads to compare ----------
def default_pair(view, peers_tbl):
    elig = view[view['Spend'] >= MIN_SPEND]
    if elig.empty:
        elig = peers_tbl[peers_tbl['Spend'] >= MIN_SPEND]
    src = elig[elig['Purchases'] >= 1].sort_values('Spend', ascending=False)
    if len(src) < 2:
        src = elig.sort_values('Spend', ascending=False)
    names = src['Ad_Name'].tolist()
    if len(names) >= 2:
        return names[0], names[1]
    all_names = peers_tbl.sort_values('Spend', ascending=False)['Ad_Name'].tolist()
    return (all_names[0], all_names[1]) if len(all_names) > 1 else (None, None)

# ---------- 9H. Put the Conversion page together ----------
def build_conversion(rows, peers):
    view, peers_tbl = conv_tables(rows, peers)
    if view.empty or view['Spend'].sum() <= 0:
        return html.Div(card('Conversion Analysis', html.Div('No data for these filters.',
                                                              style={'color': GREY, 'padding': '20px 0'})))
    tot = totals(rows)
    lpv = overall_lpv(view)
    vid_tot = totals_from_table(video_ads(view))   # video ads only, so image ads do not lower the rate

    # ---------- KPI Cards ----------
    kpis = html.Div([
        kpi_card('Spend', rm(tot['Spend'])),
        kpi_card('CPP', rm(tot['CPP']) if pd.notna(tot['CPP']) else '-', 'Spend / purchases'),
        kpi_card('Total purchases', f"{tot['Purchases']:,.0f}"),
        kpi_card('CTR Link', f2(tot['CTR_Link'], '%'), 'Link clicks / impressions'),
        kpi_card('LPV Rate', f"{lpv:.1f}%" if pd.notna(lpv) else '-', 'Landing page views / link clicks'),
        kpi_card('Purchase Value', rm(tot['Purchase_Value'])),
        kpi_card('ROAS', f2(tot['ROAS']), 'Purchase value / spend'),
        kpi_card('CPM', rm(tot['CPM']) if pd.notna(tot['CPM']) else '-', 'Cost per 1,000 impressions'),
        kpi_card('CPC', rm(tot['CPC']) if pd.notna(tot['CPC']) else '-', 'Cost per link click'),
        kpi_card('CPLPV', rm(tot['CPLPV']) if pd.notna(tot['CPLPV']) else '-', 'Cost per landing page view'),
        kpi_card('Hook Rate', f2(vid_tot['Hook_Rate'], '%'), '3-second plays / impressions (video ads)'),
        kpi_card('Hold Rate', f2(vid_tot['Hold_Rate'], '%'), 'ThruPlays / 3-second plays (video ads)'),
    ], style={
        'display': 'grid',
        'gridTemplateColumns': 'repeat(auto-fit, minmax(170px, 1fr))',
        'gap': '14px',
        'marginBottom': '16px'
    })

    lab = {'fontWeight': 'bold', 'fontSize': '13px', 'margin': '8px 0 4px 0'}
    two = {'display': 'grid', 'gridTemplateColumns': 'repeat(auto-fit, minmax(420px, 1fr))',
           'gap': '16px', 'marginBottom': '16px'}
    one = {'marginBottom': '16px'}

    # --- compare two ads ---
    order = peers_tbl.sort_values('Spend', ascending=False)['Ad_Name'].tolist()
    a0, b0 = default_pair(view, peers_tbl)
    if a0 is None:
        compare_body = grey_note(f"Need at least 2 ads with RM {MIN_SPEND}+ spend.")
    else:
        cmp_default = compare_from(peers_tbl, a0, b0, 'CPP')
        compare_body = html.Div([
            html.Div('Measure', style=lab),
            dcc.Dropdown(id='cmp-metric', options=[{'label': v, 'value': k} for k, v in CMP_METRICS.items()],
                         value='CPP', clearable=False),
            html.Div('Ad A', style=lab), dcc.Dropdown(id='cmp-a', options=order, value=a0, clearable=False),
            html.Div('Ad B', style=lab), dcc.Dropdown(id='cmp-b', options=order, value=b0, clearable=False),
            html.Div(id='cmp-out', children=cmp_default, style={'marginTop': '14px'})])

    # --- budget what-if (one ad at a time) ---
    wi_ads = peers_tbl[peers_tbl['Spend'] > 0].sort_values('Spend', ascending=False)
    wi_options = [{'label': f"{r.Ad_Name} ({plural(r.Purchases, 'purchase')})", 'value': r.Ad_Name}
                  for r in wi_ads.itertuples()]
    wi_pool = wi_ads[wi_ads['Purchases'] >= 1]
    if wi_pool.empty:
        wi_pool = wi_ads
    wi_ad0 = wi_pool['Ad_Name'].iloc[0] if len(wi_pool) else None    # default: top spender with purchases

    wi_tot = totals(peers[peers['Ad_Name'] == wi_ad0])
    max_b, step, default = slider_setup(wi_tot['Spend'])
    wi_text, wi_fig = whatif_render(wi_tot, default, WHATIF_KEY)
    whatif_body = html.Div([
        html.Div('Ad name', style=lab),
        dcc.Dropdown(id='wi-ad', options=wi_options, value=wi_ad0, clearable=False),
        html.Div('Budget for this ad, same period (RM) - drag the slider', style={**lab, 'marginTop': '14px'}),
        dcc.Slider(id='wi-budget', min=0, max=max_b, step=step, value=default, marks=None,
                   updatemode='mouseup',
                   tooltip={'placement': 'bottom', 'always_visible': True}),
        html.Div(id='wi-text', children=wi_text, style={'marginTop': '14px'}),
        dcc.Graph(id='wi-graph', figure=wi_fig, config={'displayModeBar': False}),
    ])

    return html.Div([
        kpis,
        insight_box(conversion_insights(view, peers_tbl, tot)),
        html.Div(card('Ad name vs metric (use the buttons to switch)', graph(fig_conv_switch(view)),
                      note='Green = better than the average for the ads shown, amber = worse. '
                           'Unique CTR = people who clicked / people reached. '
                           'LPV rate = landing page views / link clicks. '
                           f'Only ads with RM {MIN_SPEND}+ spend and a purchase (CPP), {MIN_IMPR_CTR:,}+ impressions (CTR), '
                           f'or {MIN_CLICKS_CVR}+ link clicks and LPV data (LPV) are shown.'), style=one),
        html.Div(card('Compare two ads: is the gap real?', compare_body,
                      note="Not affected by the Ad name filter. Uses all ads in the date range. "
                           "CPP is tested on purchases per RM spent; the other measures on their counts."), style=one),
        html.Div(card('Budget what-if simulator', whatif_body,
                      note="Pick one ad to see what to expect if its budget changes. Purchases = today's purchases x "
                           "(new budget / today's budget) ^ 0.9, so cost per purchase rises a little as the budget grows. "
                           "This is a guess, not a fact about your account: test any big budget change with a small step "
                           "first. Uses the date range above, not the Ad name filter."), style=one),
    ])

SECTION_BUILDERS['conversion'] = build_conversion

# ---------- 9I. Connect the interactive boxes (registered once, so running this cell twice is safe) ----------
if not any('cmp-out.children' in k for k in app.callback_map):
    @app.callback(Output('cmp-out', 'children'), Input('cmp-a', 'value'), Input('cmp-b', 'value'),
                  Input('cmp-metric', 'value'), State('date-range', 'start_date'), State('date-range', 'end_date'),
                  prevent_initial_call=True)
    def _cmp_cb(a, b, key, start, end):
        return compare_logic(a, b, key, start, end)

if not any('wi-graph.figure' in k for k in app.callback_map):
    # when the ad changes, reset the slider to fit that ad's spend
    @app.callback(Output('wi-budget', 'max'), Output('wi-budget', 'step'), Output('wi-budget', 'value'),
                  Input('wi-ad', 'value'),
                  State('date-range', 'start_date'), State('date-range', 'end_date'),
                  prevent_initial_call=True)
    def _wi_ad_cb(ad, start, end):
        max_b, step, default = slider_setup(ad_totals(ad, start, end)['Spend'])
        return max_b, step, default

    # redraw the text and the graph
    @app.callback(Output('wi-text', 'children'), Output('wi-graph', 'figure'),
                  Input('wi-budget', 'value'), Input('wi-ad', 'value'),
                  State('date-range', 'start_date'), State('date-range', 'end_date'),
                  prevent_initial_call=True)
    def _wi_cb(budget, ad, start, end):
        return whatif_logic(budget, ad, start, end)